# Portfolio optimisation of 20 US stocks: Markowitz, QUBO, simulated annealing, MIQP and the Ising spin-glass model

This single notebook combines the three notebooks of the project (`portfolio_optimization`, `Task2_Ising_Portfolio`, `task2_ising_portfolio_v1`). Run it **once from top to bottom** (about 5 to 10 minutes; the 30-seed experiments of Section 8 are the slowest part). The data are downloaded **only once**, in the first code cell, and the same `adj_close` table is used by every later cell.

## Objectives and where they are done

| # | Objective | Section |
|---|---|---|
| 1 | Collect daily adjusted closing prices with the Yahoo Finance API | Section 1 |
| 2 | Markowitz model on the stocks, giving the values (mean returns $\mu$, covariance $\Sigma$, risk aversion $\lambda$) used in the QUBO | Sections 2, 3, 4 and 5 |
| 3 | Classical solvers on the QUBO: simulated annealing and MIQP | Sections 6 and 7 |
| 4 | Ising spin-glass model and its optimisation | Section 8 |
| 5 | Comparison of simulated annealing / MIQP with the Ising model and a written statement | Section 9 |

## Sections

1. **Data**: download of the adjusted closing prices (one cell).
2. **Return, volatility and Sharpe-ratio charts**: cumulative return, risk-return-Sharpe, rolling Sharpe ratio and bootstrap confidence intervals.
3. **Markowitz model**: efficient frontier, minimum-variance and maximum-Sharpe portfolios.
4. **Settings, inputs and capped Markowitz solutions**: one shared settings cell, $\mu$, $\Sigma$, and the reference portfolios used in all later charts.
5. **QUBO formulation**: risk aversion $\lambda$, the QUBO matrix, the energy landscape and the Monte-Carlo estimate of the penalty $P$.
6. **Classical solver 1, simulated annealing**: own simulated annealer for $k=1\ldots 20$, choice of $k$, and the D-Wave classical simulated-annealing sampler.
7. **Classical solver 2, MIQP**: mixed-integer quadratic programme with free weights.
8. **Ising spin-glass model**: QUBO to Ising, validation, exact ground state, Ising simulated annealing, 30-seed experiments, sensitivity and charts.
9. **Comparison table and conclusions**.

## Conventions used everywhere

* $\mu$ = annualised mean of daily simple returns (252 trading days), $\Sigma$ = annualised covariance of daily simple returns, both from the **same full period**, computed once.
* Only days on which **every** stock traded are used for $\mu$ and $\Sigma$.
* Portfolio objective: $f(w)=\tfrac{\lambda}{2}\,w^\top\Sigma w-\mu^\top w$. Every method below minimises exactly this objective, so all results are comparable.
* Figures and tables are saved in the folder `portfolio_20stocks_outputs`.

Packages needed: `numpy pandas scipy matplotlib yfinance dimod dwave-samplers` and an MIQP solver (`gurobipy` or `pyscipopt`). Missing packages are installed automatically by the helper `_ensure` below.

## Section 1. Data: daily adjusted closing prices (Yahoo Finance API)

**What the next cell does (objective 1)**

* imports every library used in the notebook and defines the helper `_ensure`, which installs a package if it is missing;
* downloads the daily prices of the 20 stocks with `yfinance` (2000-01-01 to 2025-12-31; the `end` date of `yfinance` is exclusive), with up to 3 retries per stock;
* keeps the **adjusted close** (`Adj Close`: split- *and* dividend-adjusted) of every stock and puts them in one table `adj_close` (rows = dates, columns = tickers);
* prints a summary (rows, first and last date, missing values per ticker) and saves the table once as `us_stock_data/adj_close_wide.csv`.

This is the **only** cell that contacts Yahoo Finance. Every later cell uses the variable `adj_close` that is created here.

Note: Yahoo Finance writes share classes with a hyphen (`BRK.B` is `BRK-B`).

In [ ]:
# SECTION 1 | DATA - the only cell that downloads data. Creates the table `adj_close` used by the whole notebook.
import os, sys, time, subprocess, importlib, itertools, warnings
from math import comb
from itertools import combinations, islice

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.colors import TwoSlopeNorm
from scipy.optimize import minimize
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

def _ensure(module, pip_name):
    """import a package; if it is missing, install it into the Python that runs THIS notebook, then import it."""
    try:
        return importlib.import_module(module)
    except ImportError:
        print(f"'{module}' not found in {sys.executable} -> installing '{pip_name}' ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        importlib.invalidate_caches()
        return importlib.import_module(module)

yf = _ensure("yfinance", "yfinance")

START_DATE = "2000-01-01"
END_DATE = "2026-01-01"          # yfinance 'end' is exclusive
DATA_DIR = "us_stock_data"
OUT_DIR = "portfolio_20stocks_outputs"        # every figure / table of the notebook is saved here
MAX_RETRIES = 3
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

def savefig(name, dpi=300):
    plt.savefig(f"{OUT_DIR}/{name}.png", dpi=dpi, bbox_inches="tight")

# Yahoo Finance uses a hyphen for share classes: BRK.B -> BRK-B
STOCKS = {
    "MSFT":  "Microsoft Corporation",
    "AAPL":  "Apple Inc.",
    "NVDA":  "NVIDIA Corporation",
    "AMZN":  "Amazon.com Inc.",
    "INTC":  "Intel Corporation",
    "WMT":   "Walmart Inc.",
    "KO":    "The Coca-Cola Company",
    "DIS":   "The Walt Disney Company",
    "PG":    "The Procter & Gamble Company",
    "IBM":   "International Business Machines",
    "MCD":   "McDonald's Corporation",
    "JPM":   "JPMorgan Chase & Co.",
    "GS":    "The Goldman Sachs Group, Inc.",
    "MS":    "Morgan Stanley",
    "AXP":   "American Express Company",
    "BAC":   "Bank of America Corporation",
    "JNJ":   "Johnson & Johnson",
    "XOM":   "ExxonMobil Corporation",
    "UNH":   "UnitedHealth Group Incorporated",
    "BRK-B": "Berkshire Hathaway Inc. (Class B)",
}

def download_adj_close(ticker):
    """Daily adjusted close of one ticker (auto_adjust=False keeps the 'Adj Close' column), with retries."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            df = yf.download(ticker, start=START_DATE, end=END_DATE, interval="1d",
                             auto_adjust=False, progress=False, threads=False)
            if df is not None and not df.empty:
                if isinstance(df.columns, pd.MultiIndex):          # newer yfinance versions return MultiIndex columns
                    df.columns = df.columns.get_level_values(0)
                df.index = pd.to_datetime(df.index)
                df.index.name = "Date"
                return df["Adj Close"].dropna(how="all")
            print(f"  [{ticker}] empty result (attempt {attempt}/{MAX_RETRIES})")
        except Exception as e:
            print(f"  [{ticker}] error: {e} (attempt {attempt}/{MAX_RETRIES})")
        time.sleep(2 * attempt)
    return pd.Series(dtype=float)

print(f"Downloading {len(STOCKS)} stocks from {START_DATE} to {END_DATE}\n")
series, summary_rows, failed = {}, [], []
for i, (ticker, name) in enumerate(STOCKS.items(), start=1):
    print(f"[{i:02d}/{len(STOCKS)}] {ticker} - {name}")
    s_ = download_adj_close(ticker)
    if s_.empty:
        failed.append(ticker)
        continue
    series[ticker] = s_
    summary_rows.append({"Ticker": ticker, "Name": name, "Rows": len(s_), "First_Date": s_.index.min().date(),
                         "Last_Date": s_.index.max().date(), "Missing_Adj_Close": int(s_.isna().sum())})
    print(f"  -> {len(s_)} rows ({s_.index.min().date()} to {s_.index.max().date()})")
    time.sleep(0.5)                                                # be polite to Yahoo

if not series:
    raise SystemExit("No data downloaded. Check your internet connection.")

adj_close = pd.DataFrame(series).sort_index()                     # Date x Ticker matrix of adjusted close prices
adj_close.to_csv(f"{DATA_DIR}/adj_close_wide.csv")                # saved once for your records; not read again
print("\n" + "=" * 70)
display(pd.DataFrame(summary_rows))
print("=" * 70)
print(f"Successful: {len(series)}/{len(STOCKS)}" + (f"   |   Failed: {failed}" if failed else ""))
print(f"adj_close: {adj_close.shape[0]} dates x {adj_close.shape[1]} tickers")

## Section 2. Return, volatility and Sharpe-ratio charts

These charts describe each stock on its own, before any optimisation. They use the table `adj_close` of Section 1 (no new download). The descriptive charts use a risk-free rate `RF_DESC = 2%` for the Sharpe ratio; the optimisation sections (4 onwards) use `RF = 0`, as in the original notebook.

### 2.1 Cumulative return of the 20 stocks

**What the next cell does**

* builds the *wealth index* $W_t = P_t / P_{t_0}$ (value of 1 unit invested at the first available price of each stock) and the cumulative return $(W_t-1)\cdot 100\%$;
* computes the total return and the CAGR $=\left(W_T\right)^{1/\text{years}}-1$ of each stock;
* draws three figures: (1) all stocks on one log-scale chart with end labels, (2) one small panel per stock, (3) ranking by total return and by CAGR.

In [ ]:
# SECTION 2.1 | Cumulative return of the 20 stocks
# adj_close comes from Section 1 (no new download)

# Wealth index: value of $1 invested at each stock's first available price
wealth = adj_close / adj_close.apply(lambda s: s.dropna().iloc[0])
cum_ret = (wealth - 1) * 100                     # cumulative return in %
final_ret = cum_ret.ffill().iloc[-1].sort_values(ascending=False)
years = (adj_close.index[-1] - adj_close.index[0]).days / 365.25
cagr = (wealth.ffill().iloc[-1] ** (1 / years) - 1) * 100

# Sector colours (makes the chart easier to interpret)
sector = {
    "MSFT": "Technology", "AAPL": "Technology", "NVDA": "Technology",
    "INTC": "Technology", "IBM": "Technology",
    "AMZN": "Consumer", "WMT": "Consumer", "KO": "Consumer",
    "DIS": "Consumer", "PG": "Consumer", "MCD": "Consumer",
    "JPM": "Financials", "GS": "Financials", "MS": "Financials",
    "AXP": "Financials", "BAC": "Financials", "BRK-B": "Financials",
    "JNJ": "Healthcare", "UNH": "Healthcare",
    "XOM": "Energy",
}
sector_color = {"Technology": "#1f77b4", "Consumer": "#ff7f0e", "Financials": "#2ca02c",
                "Healthcare": "#d62728", "Energy": "#7f7f7f"}
col = {t: sector_color[sector[t]] for t in wealth.columns}

plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

# Y-axis: log scale on wealth index, tick labels shown as cumulative return (%)
def ret_formatter():
    return mtick.FuncFormatter(lambda w, _: f"{(w - 1) * 100:,.0f}%")

ticks = np.array([0.25, 0.5, 1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000])

# FIGURE 1: Cumulative return (log scale) with direct end-labels

fig, ax = plt.subplots(figsize=(16, 9), dpi=120)

for t in wealth.columns:
    ax.plot(wealth.index, wealth[t], color=col[t], lw=1.4, alpha=0.9)

ax.set_yscale("log")
ymin, ymax = wealth.min().min() * 0.8, wealth.max().max() * 1.6
ax.set_ylim(ymin, ymax)
ax.set_yticks(ticks[(ticks >= ymin) & (ticks <= ymax)])
ax.yaxis.set_major_formatter(ret_formatter())
ax.yaxis.set_minor_formatter(mtick.NullFormatter())
ax.axhline(1, color="black", lw=0.9, ls="--")          # break-even line
ax.grid(True, which="major", alpha=0.35)

# Direct labels at the right edge (spread out so they don't overlap)
last = wealth.ffill().iloc[-1].sort_values()
pos, min_gap, prev = {}, 0.055, None
for t, v in last.items():
    y = np.log10(v)
    y = y if prev is None or y - prev >= min_gap else prev + min_gap
    pos[t], prev = y, y
x_end = wealth.index[-1] + pd.Timedelta(days=90)
for t, y in pos.items():
    ax.annotate(f"{t}  {final_ret[t]:,.0f}%", xy=(x_end, 10 ** y), va="center",
                fontsize=9, color=col[t], fontweight="bold", annotation_clip=False)

ax.set_xlim(wealth.index[0], wealth.index[-1] + pd.Timedelta(days=30))
ax.set_title("Cumulative Return (%) of 20 US Stocks, 2000-2025 (log scale, dividend-adjusted)",
             fontsize=15, fontweight="bold", pad=12)
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative Return (%)  [log scale]")

handles = [plt.Line2D([0], [0], color=c, lw=3, label=s) for s, c in sector_color.items()]
ax.legend(handles=handles, loc="upper left", title="Sector", frameon=True)
fig.subplots_adjust(right=0.88)
savefig("cumulative_return_log")
plt.show()


# FIGURE 2: Small multiples (one panel per stock)

order = list(final_ret.index)
fig, axes = plt.subplots(5, 4, figsize=(18, 14), sharex=True, dpi=120)

for ax, t in zip(axes.flat, order):
    w = wealth[t].dropna()
    ax.plot(w.index, w, color=col[t], lw=1.2)
    ax.axhline(1, color="black", lw=0.7, ls="--")
    ax.fill_between(w.index, 1, w, where=(w >= 1), color=col[t], alpha=0.15)
    ax.fill_between(w.index, 1, w, where=(w < 1), color="red", alpha=0.15)
    ax.set_yscale("log")
    lo, hi = w.min() * 0.8, w.max() * 1.5
    ax.set_ylim(lo, hi)
    tk = ticks[(ticks >= lo) & (ticks <= hi)]
    ax.set_yticks(tk)
    ax.yaxis.set_major_formatter(ret_formatter())
    ax.yaxis.set_minor_formatter(mtick.NullFormatter())
    ax.set_title(f"{t}  ({sector[t]})", fontsize=11, fontweight="bold", color=col[t])
    ax.text(0.03, 0.88, f"Total: {final_ret[t]:,.0f}%\nCAGR: {cagr[t]:.1f}%",
            transform=ax.transAxes, fontsize=9, va="top",
            bbox=dict(boxstyle="round", fc="white", ec="lightgrey"))
    ax.grid(True, alpha=0.3)

fig.suptitle("Cumulative Return (%) by Stock, 2000-2025 (log scale, ordered by total return)",
             fontsize=16, fontweight="bold", y=0.995)
fig.supxlabel("Date")
fig.supylabel("Cumulative Return (%)")
plt.tight_layout()
savefig("cumulative_return_small_multiples")
plt.show()


# FIGURE 3: Total return and CAGR ranking

fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 8), dpi=120)

fr = final_ret.sort_values()
a1.barh(fr.index, fr.values, color=[col[t] for t in fr.index])
a1.set_xscale("log")
a1.set_title("Total Cumulative Return (%)  [log scale]", fontweight="bold")
a1.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}%"))
for i, v in enumerate(fr.values):
    a1.text(v * 1.08, i, f"{v:,.0f}%", va="center", fontsize=9)
a1.set_xlim(right=fr.max() * 4)

cg = cagr.sort_values()
a2.barh(cg.index, cg.values, color=[col[t] for t in cg.index])
a2.set_title("Compound Annual Growth Rate (CAGR, %)", fontweight="bold")
a2.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
for i, v in enumerate(cg.values):
    a2.text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=9)
a2.set_xlim(right=cg.max() * 1.15)

for a in (a1, a2):
    a.grid(True, axis="x", alpha=0.3)
handles = [plt.Line2D([0], [0], color=c, lw=6, label=s) for s, c in sector_color.items()]
a2.legend(handles=handles, loc="lower right", title="Sector")
plt.tight_layout()
savefig("return_ranking")
plt.show()

# Summary table
summary = pd.DataFrame({"Sector": pd.Series(sector), "Total Return (%)": final_ret.round(1),
                        "CAGR (%)": cagr.round(2)}).loc[order]
summary

### 2.2 Return, volatility and Sharpe ratio of each stock

**What the next cell does**

* computes from the daily simple returns $r_t$ (each stock uses its own trading history): annualised return $\bar r\cdot 252$, annualised volatility $s_r\sqrt{252}$ and the Sharpe ratio
$$\text{Sharpe}=\frac{\bar r\cdot 252-\text{RF\_DESC}}{s_r\sqrt{252}};$$
* Figure 1: return versus volatility, coloured by Sharpe ratio, with iso-Sharpe lines;
* Figure 2: ranking of return, volatility and Sharpe ratio;
* Figure 3: correlation matrix of the daily returns (this is the information that enters $\Sigma$);
* the last line prints the table of the metrics.

In [ ]:
# SECTION 2.2 | Return, volatility and Sharpe ratio of each stock
# Settings
RF_DESC = 0.02              # annual risk-free rate (2%); change to suit your study
TRADING_DAYS = 252


# adj_close comes from Section 1 (no new download)

# Daily simple ret_all (no forward-filling, so each stock uses only its own trading history)
ret_all = adj_close.pct_change(fill_method=None)


# Risk / return metrics

ann_return = ret_all.mean() * TRADING_DAYS                    # annualised mean return
ann_vol = ret_all.std() * np.sqrt(TRADING_DAYS)               # annualised volatility
sharpe = (ann_return - RF_DESC) / ann_vol                          # Sharpe ratio

years = (adj_close.index[-1] - adj_close.index[0]).days / 365.25
first_price = adj_close.apply(lambda s: s.dropna().iloc[0])
cagr = ((adj_close.ffill().iloc[-1] / first_price) ** (1 / years) - 1)

metrics = pd.DataFrame({
    "Ann. Return (%)": ann_return * 100,
    "Ann. Volatility (%)": ann_vol * 100,
    "Sharpe Ratio": sharpe,
    "CAGR (%)": cagr * 100,
}).sort_values("Sharpe Ratio", ascending=False)

plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

# FIGURE 1: Return vs Volatility, coloured by Sharpe ratio

fig, ax = plt.subplots(figsize=(14, 9), dpi=120)

x = metrics["Ann. Volatility (%)"]
y = metrics["Ann. Return (%)"]
s = metrics["Sharpe Ratio"]

norm = TwoSlopeNorm(vmin=min(s.min(), -0.01), vcenter=s.median(), vmax=s.max())
sc = ax.scatter(x, y, c=s, cmap="RdYlGn", norm=norm, s=350,
                edgecolor="black", linewidth=0.8, zorder=3)

for t in metrics.index:
    ax.annotate(t, (x[t], y[t]), xytext=(0, 14), textcoords="offset points",
                ha="center", fontsize=9, fontweight="bold")

# Iso-Sharpe lines: return = RF_DESC + Sharpe * volatility
xs = np.linspace(0, x.max() * 1.15, 100)
for sh in [0.25, 0.5, 0.75, 1.0]:
    ax.plot(xs, (RF_DESC + sh * xs / 100) * 100, ls="--", lw=0.9, color="grey", alpha=0.7, zorder=1)
    ax.text(xs[-1], (RF_DESC + sh * xs[-1] / 100) * 100, f" SR={sh}", fontsize=8, color="grey", va="center")

ax.axhline(RF_DESC * 100, color="black", lw=0.8, ls=":")
ax.text(x.min() * 0.9, RF_DESC * 100 + 0.3, f"Risk-free = {RF_DESC*100:.1f}%", fontsize=8)

cbar = plt.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label("Sharpe Ratio")

ax.set_xlim(x.min() * 0.85, x.max() * 1.2)
ax.set_xlabel("Annualised Volatility (%)")
ax.set_ylabel("Annualised Return (%)")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.grid(True, alpha=0.3)
ax.set_title("Return vs Volatility vs Sharpe Ratio: 20 US Stocks (2000-2025)",
             fontsize=15, fontweight="bold", pad=12)
plt.tight_layout()
savefig("return_vs_volatility_sharpe")
plt.show()


# FIGURE 2: Ranking of return, volatility and Sharpe ratio

fig, axes = plt.subplots(1, 3, figsize=(18, 8), dpi=120, sharey=True)
order = metrics.sort_values("Sharpe Ratio").index

panels = [
    ("Ann. Return (%)", "Annualised Return (%)", "#2ca02c"),
    ("Ann. Volatility (%)", "Annualised Volatility (%)", "#d62728"),
    ("Sharpe Ratio", f"Sharpe Ratio (rf = {RF_DESC*100:.1f}%)", "#1f77b4"),
]
for a, (colname, title, c) in zip(axes, panels):
    vals = metrics.loc[order, colname]
    a.barh(order, vals, color=c, alpha=0.85)
    for i, v in enumerate(vals):
        a.text(v + vals.max() * 0.01, i, f"{v:.2f}" if colname == "Sharpe Ratio" else f"{v:.1f}%",
               va="center", fontsize=9)
    a.set_title(title, fontweight="bold")
    a.grid(True, axis="x", alpha=0.3)
    a.set_xlim(right=vals.max() * 1.15)

plt.suptitle("Risk-Return Ranking (sorted by Sharpe ratio)", fontsize=15, fontweight="bold")
plt.tight_layout()
savefig("risk_return_ranking")
plt.show()


# FIGURE 3: Correlation of daily ret_all (useful for portfolio work)

corr = ret_all.corr()
fig, ax = plt.subplots(figsize=(11, 9), dpi=120)
im = ax.imshow(corr, cmap="coolwarm", vmin=-0.2, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
plt.colorbar(im, ax=ax, label="Correlation")
ax.set_title("Correlation Matrix of Daily Returns", fontsize=14, fontweight="bold")
plt.tight_layout()
savefig("correlation_matrix")
plt.show()

metrics.round(2)

### 2.3 Rolling Sharpe ratio and bootstrap confidence intervals

**What the next cell does**

* **Part 1, rolling Sharpe ratio**: for each calendar year-end it takes the previous 3 years of daily returns (a stock needs at least 80 % of the trading days of the window) and computes the Sharpe ratio. Figure 1 is the heat map (stocks sorted by mean rolling Sharpe), Figure 2 one panel per stock (red dashed line = full-period Sharpe), Figure 3 the stability of return, volatility and Sharpe over time.
* **Part 2, block bootstrap**: 1000 circular block-bootstrap resamples of the daily returns (blocks of 21 trading days keep the volatility clustering) give a 95 % confidence interval for the full-period Sharpe ratio of each stock (Figure 4, green = interval excludes 0).
* the last line prints a table with the rolling-Sharpe statistics and the confidence intervals.

In [ ]:
# SECTION 2.3 | Rolling Sharpe ratio and bootstrap confidence intervals
RF_DESC = 0.02               # annual risk-free rate (same as in Section 2.2)
TD = 252                # trading days per year
WINDOW_YEARS = 3        # rolling window length
N_BOOT = 1000           # bootstrap resamples
BLOCK = 21              # block length (~1 trading month) to preserve volatility clustering
SEED = 42


# adj_close comes from Section 1 (no new download)
ret_all = adj_close.pct_change(fill_method=None)

def sharpe_fn(r):
    return (r.mean() * TD - RF_DESC) / (r.std() * np.sqrt(TD))

def span_label(end_year):
    """Label a window by its full span, e.g. end year 2002 -> '2000-02'."""
    start_year = end_year - WINDOW_YEARS + 1
    return f"{start_year}-{str(end_year)[-2:]}"

plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})


# PART 1: Rolling 3-year windows, stepped yearly (window ends at each year-end)

end_years = range(ret_all.index[0].year + WINDOW_YEARS - 1, ret_all.index[-1].year + 1)
roll_sharpe, roll_ret, roll_vol = {}, {}, {}

for y in end_years:
    end = ret_all.index[ret_all.index.year == y].max()
    start = end - pd.DateOffset(years=WINDOW_YEARS)
    w = ret_all.loc[(ret_all.index > start) & (ret_all.index <= end)]
    enough = w.count() >= 0.8 * TD * WINDOW_YEARS          # need at least 80% of the window
    roll_ret[y] = pd.Series({t: w[t].mean() * TD if enough[t] else np.nan for t in w.columns})
    roll_vol[y] = pd.Series({t: w[t].std() * np.sqrt(TD) if enough[t] else np.nan for t in w.columns})
    roll_sharpe[y] = pd.Series({t: sharpe_fn(w[t].dropna()) if enough[t] else np.nan for t in w.columns})

roll_sharpe = pd.DataFrame(roll_sharpe)     # stocks x window-end-year
roll_ret = pd.DataFrame(roll_ret) * 100
roll_vol = pd.DataFrame(roll_vol) * 100

order = roll_sharpe.mean(axis=1).sort_values(ascending=False).index
roll_sharpe = roll_sharpe.loc[order]

# Window labels (full span) and x positions
win_labels = [span_label(y) for y in roll_sharpe.columns]
pos = np.arange(len(win_labels))

# Figure 1: heatmap of rolling Sharpe
fig, ax = plt.subplots(figsize=(17, 9), dpi=120)
vmax = np.nanmax(np.abs(roll_sharpe.values))
im = ax.imshow(roll_sharpe.values, aspect="auto", cmap="RdYlGn",
               norm=TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax))
ax.set_xticks(pos)
ax.set_xticklabels(win_labels, rotation=90)
ax.set_yticks(range(len(roll_sharpe)))
ax.set_yticklabels(roll_sharpe.index)
for i in range(roll_sharpe.shape[0]):
    for j in range(roll_sharpe.shape[1]):
        v = roll_sharpe.iat[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=7)
plt.colorbar(im, ax=ax, pad=0.01, label="Sharpe ratio")
ax.set_xlabel(f"{WINDOW_YEARS}-year window (start year to end year)")
ax.set_title(f"Rolling {WINDOW_YEARS}-Year Sharpe Ratio (stocks sorted by mean rolling Sharpe)",
             fontsize=15, fontweight="bold", pad=12)
for s in ax.spines.values():
    s.set_visible(False)
plt.tight_layout()
savefig("rolling_sharpe_heatmap")
plt.show()

# Figure 2: rolling Sharpe small multiples
full_sharpe = ret_all.apply(lambda s: sharpe_fn(s.dropna()))
fig, axes = plt.subplots(5, 4, figsize=(18, 15), sharex=True, sharey=True, dpi=120)
for ax, t in zip(axes.flat, order):
    s = roll_sharpe.loc[t].values
    ax.plot(pos, s, color="#1f77b4", lw=1.6, marker="o", ms=3)
    ax.axhline(0, color="black", lw=0.8)
    ax.axhline(full_sharpe[t], color="red", ls="--", lw=1, label="Full-period Sharpe")
    ax.fill_between(pos, 0, s, where=(s >= 0), color="green", alpha=0.15)
    ax.fill_between(pos, 0, s, where=(s < 0), color="red", alpha=0.15)
    ax.set_title(t, fontsize=11, fontweight="bold")
    ax.text(0.03, 0.05, f"mean {np.nanmean(s):.2f} | std {np.nanstd(s, ddof=1):.2f}",
            transform=ax.transAxes, fontsize=8,
            bbox=dict(boxstyle="round", fc="white", ec="lightgrey"))
    ax.set_xticks(pos[::3])
    ax.set_xticklabels(win_labels[::3], rotation=90, fontsize=8)
    ax.grid(True, alpha=0.3)
axes.flat[0].legend(fontsize=8, loc="upper left")
fig.suptitle(f"Rolling {WINDOW_YEARS}-Year Sharpe Ratio by Stock (red dashed = full-period Sharpe)",
             fontsize=16, fontweight="bold", y=0.995)
fig.supxlabel(f"{WINDOW_YEARS}-year window (start year to end year)")
fig.supylabel("Sharpe ratio")
plt.tight_layout()
savefig("rolling_sharpe_small_multiples")
plt.show()

# Figure 3: stability of return / volatility / Sharpe over time
fig, axes = plt.subplots(1, 3, figsize=(19, 6.5), dpi=120)
for a, (df, title) in zip(axes, [(roll_ret, "Rolling annualised return (%)"),
                                 (roll_vol, "Rolling annualised volatility (%)"),
                                 (roll_sharpe, "Rolling Sharpe ratio")]):
    for t in df.index:
        a.plot(pos, df.loc[t].values, color="grey", lw=0.8, alpha=0.5)
    a.plot(pos, df.median().values, color="red", lw=2.5, label="Cross-sectional median")
    a.axhline(0, color="black", lw=0.7)
    a.set_title(title, fontweight="bold")
    a.set_xticks(pos[::2])
    a.set_xticklabels(win_labels[::2], rotation=90, fontsize=8)
    a.set_xlabel(f"{WINDOW_YEARS}-year window (start year to end year)")
    a.grid(True, alpha=0.3)
axes[0].legend()
plt.tight_layout()
savefig("rolling_ret_vol_sharpe")
plt.show()

# PART 2: Block-bootstrap confidence intervals for the full-period Sharpe ratio

rng = np.random.default_rng(SEED)

def block_bootstrap_sharpe(r, n_boot, block, rng):
    """Circular block bootstrap of daily ret_all; ret_all the distribution of Sharpe ratios."""
    r = r.dropna().values
    n = len(r)
    n_blocks = int(np.ceil(n / block))
    starts = rng.integers(0, n, size=(n_boot, n_blocks))
    idx = ((starts[:, :, None] + np.arange(block)).reshape(n_boot, -1)[:, :n]) % n
    samp = r[idx]
    return (samp.mean(axis=1) * TD - RF_DESC) / (samp.std(axis=1, ddof=1) * np.sqrt(TD))

boot_rows = {}
for t in ret_all.columns:
    dist = block_bootstrap_sharpe(ret_all[t], N_BOOT, BLOCK, rng)
    boot_rows[t] = {"Sharpe": full_sharpe[t],
                    "CI Low (2.5%)": np.percentile(dist, 2.5),
                    "CI High (97.5%)": np.percentile(dist, 97.5)}
boot = pd.DataFrame(boot_rows).T.sort_values("Sharpe")
boot["Significant (CI > 0)"] = boot["CI Low (2.5%)"] > 0

# Figure 4: forest plot
fig, ax = plt.subplots(figsize=(11, 9), dpi=120)
y = np.arange(len(boot))
colors = np.where(boot["Significant (CI > 0)"], "#2ca02c", "#7f7f7f")
ax.hlines(y, boot["CI Low (2.5%)"], boot["CI High (97.5%)"], color=colors, lw=2.5)
ax.scatter(boot["Sharpe"], y, color=colors, s=70, zorder=3, edgecolor="black")
ax.axvline(0, color="black", lw=0.9, ls="--")
ax.set_yticks(y)
ax.set_yticklabels(boot.index)
for yi, (lo, hi, sh) in enumerate(zip(boot["CI Low (2.5%)"], boot["CI High (97.5%)"], boot["Sharpe"])):
    ax.text(hi + 0.02, yi, f"{sh:.2f}  [{lo:.2f}, {hi:.2f}]", va="center", fontsize=8)
ax.set_xlim(boot["CI Low (2.5%)"].min() - 0.1, boot["CI High (97.5%)"].max() + 0.5)
ax.set_xlabel("Sharpe ratio")
ax.set_title(f"Full-Period Sharpe Ratio with 95% Block-Bootstrap CI\n"
             f"({N_BOOT} resamples, {BLOCK}-day blocks; green = CI excludes 0)",
             fontsize=14, fontweight="bold")
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
savefig("sharpe_bootstrap_ci")
plt.show()

stability = pd.DataFrame({
    "Mean Rolling Sharpe": roll_sharpe.mean(axis=1),
    "Std Rolling Sharpe": roll_sharpe.std(axis=1),
    "Min": roll_sharpe.min(axis=1),
    "Max": roll_sharpe.max(axis=1),
    "% Windows Sharpe > 0": (roll_sharpe > 0).sum(axis=1) / roll_sharpe.count(axis=1) * 100,
})
summary = stability.join(boot).sort_values("Mean Rolling Sharpe", ascending=False).round(2)
summary

## Section 3. Markowitz mean-variance model

For $n$ stocks with annualised mean returns $\mu\in\mathbb R^n$ and covariance matrix $\Sigma\in\mathbb R^{n\times n}$, a portfolio is a weight vector $w$ (long-only and fully invested: $w_i\ge 0$, $\sum_i w_i=1$) with

$$\text{expected return } R(w)=\mu^\top w,\qquad \text{variance } \sigma^2(w)=w^\top\Sigma w,\qquad \text{Sharpe}(w)=\frac{R(w)-\text{RF}}{\sigma(w)} .$$

* **Minimum-variance portfolio**: $\min_w\; w^\top\Sigma w$.
* **Maximum-Sharpe (tangency) portfolio**: $\max_w\; \dfrac{\mu^\top w-\text{RF}}{\sqrt{w^\top\Sigma w}}$.
* **Efficient frontier**: for a grid of target returns $R^\star$, $\min_w\; w^\top\Sigma w$ subject to $\mu^\top w=R^\star$.

**What the next cell does (objective 2)**

* uses the dates on which every stock traded (`dropna`), and computes $\mu$ (daily mean $\times 252$) and $\Sigma$ (daily covariance $\times 252$);
* solves the three problems above with SLSQP (long-only, weights between 0 and 1, `RF = 0`);
* draws 10 000 random long-only portfolios (Dirichlet weights) as the feasible set, the efficient frontier, the individual stocks, and the minimum-variance and maximum-Sharpe portfolios (chart: expected return versus variance);
* prints the statistics of the stocks, of the two optimal portfolios and their weights.

In [ ]:
# SECTION 3 | Markowitz mean-variance model (efficient frontier, minimum-variance and maximum-Sharpe portfolios)
ANALYSIS_START = "2000-01-01"
ANALYSIS_END   = "2026-01-01"
TD = 252
N_RANDOM_PORT = 10000
N_FRONTIER = 60
RF = 0.0                        # risk-free rate for max-Sharpe (0 to match your original code)

# adj_close comes from Section 1 (no new download)
prices = adj_close.loc[ANALYSIS_START:ANALYSIS_END].sort_index()

daily_returns = prices.pct_change(fill_method=None).dropna(how="all")
daily_returns = daily_returns.dropna(axis=1, how="all")
daily_returns = daily_returns.dropna()          # same dates for every stock

assets = list(daily_returns.columns)
n = len(assets)
returns_array = (daily_returns.mean() * TD).to_numpy()
covariance_array = (daily_returns.cov() * TD).to_numpy()
print(f"{n} stocks | {daily_returns.index[0].date()} to {daily_returns.index[-1].date()} "
      f"| {len(daily_returns)} daily observations")


# Portfolio functions and constraints (long-only, fully invested)

def port_return(w):
    return float(w @ returns_array)

def port_variance(w):
    return float(w @ covariance_array @ w)

def port_vol(w):
    return np.sqrt(max(port_variance(w), 0.0))

def neg_sharpe(w):
    v = port_vol(w)
    return -(port_return(w) - RF) / v if v > 0 else 0.0

budget = {"type": "eq", "fun": lambda w: np.sum(w) - 1.0}
bounds = tuple((0.0, 1.0) for _ in range(n))
w0 = np.full(n, 1.0 / n)

# Minimum-variance and maximum-Sharpe portfolios

res_minvar = minimize(port_variance, w0, method="SLSQP", bounds=bounds,
                      constraints=(budget,), options={"maxiter": 1000, "ftol": 1e-12})
if not res_minvar.success:
    raise RuntimeError(f"Minimum-variance optimization failed: {res_minvar.message}")
w_minvar = res_minvar.x
ret_minvar, var_minvar = port_return(w_minvar), port_variance(w_minvar)

res_maxsh = minimize(neg_sharpe, w0, method="SLSQP", bounds=bounds,
                     constraints=(budget,), options={"maxiter": 1000, "ftol": 1e-12})
if not res_maxsh.success:
    raise RuntimeError(f"Maximum-Sharpe optimization failed: {res_maxsh.message}")
w_maxsh = res_maxsh.x
ret_maxsh, var_maxsh = port_return(w_maxsh), port_variance(w_maxsh)
sharpe_maxsh = (ret_maxsh - RF) / np.sqrt(var_maxsh)

# Efficient frontier (fix target return, minimise variance)

targets = np.linspace(ret_minvar, float(returns_array.max()), N_FRONTIER)
f_ret, f_var = [], []
for tr in targets:
    cons = (budget,
            {"type": "eq", "fun": lambda w, tr=tr: port_return(w) - tr})
    r = minimize(port_variance, w0, method="SLSQP", bounds=bounds, constraints=cons,
                 options={"maxiter": 1000, "ftol": 1e-12})
    if r.success:
        f_ret.append(port_return(r.x))
        f_var.append(port_variance(r.x))
f_ret, f_var = np.asarray(f_ret), np.asarray(f_var)

# Random long-only portfolios (feasible set)

rng = np.random.default_rng(42)
rw = rng.dirichlet(np.ones(n), size=N_RANDOM_PORT)
r_ret = rw @ returns_array
r_var = np.einsum("ij,jk,ik->i", rw, covariance_array, rw)
r_sharpe = (r_ret - RF) / np.sqrt(np.maximum(r_var, 1e-15))

# Plot: expected return vs variance

plt.figure(figsize=(13, 8))
sc = plt.scatter(r_var, r_ret, c=r_sharpe, cmap="viridis", s=8, alpha=0.25,
                 label="Feasible portfolios")
plt.scatter(np.diag(covariance_array), returns_array, color="black", s=65, zorder=4,
            label="Individual stocks")
for name, v, r in zip(assets, np.diag(covariance_array), returns_array):
    plt.annotate(name, (v, r), xytext=(5, 5), textcoords="offset points", fontsize=9)

if len(f_var) > 0:
    o = np.argsort(f_var)
    plt.plot(f_var[o], f_ret[o], color="red", lw=2.5, label="Efficient frontier")

plt.scatter(var_minvar, ret_minvar, color="blue", marker="*", s=220, zorder=5,
            label="Minimum-variance portfolio")
plt.scatter(var_maxsh, ret_maxsh, color="darkorange", marker="*", s=220, zorder=5,
            label="Maximum-Sharpe portfolio")

plt.xlabel("Annualized Portfolio Variance")
plt.ylabel("Annualized Expected Return")
plt.title(f"Markowitz Model: Expected Return vs Variance "
          f"({daily_returns.index[0].date()} to {daily_returns.index[-1].date()})")
plt.colorbar(sc, label="Sharpe Ratio")
plt.grid(True, alpha=0.3)
plt.legend(loc="best")
plt.tight_layout()
savefig("markowitz_frontier")
plt.show()

ind_stats = pd.DataFrame({
    "Expected Return (%)": returns_array * 100,
    "Variance": np.diag(covariance_array),
    "Volatility (%)": np.sqrt(np.diag(covariance_array)) * 100,
}, index=assets).sort_values("Expected Return (%)", ascending=False)

port_stats_tab = pd.DataFrame({
    "Portfolio": ["Minimum variance", "Maximum Sharpe"],
    "Expected Return (%)": [ret_minvar * 100, ret_maxsh * 100],
    "Variance": [var_minvar, var_maxsh],
    "Volatility (%)": [np.sqrt(var_minvar) * 100, np.sqrt(var_maxsh) * 100],
    "Sharpe Ratio": [(ret_minvar - RF) / np.sqrt(var_minvar), sharpe_maxsh],
})

weights = pd.DataFrame({
    "Minimum-variance weight (%)": w_minvar * 100,
    "Maximum-Sharpe weight (%)": w_maxsh * 100,
}, index=assets)
weights = weights[(weights > 0.01).any(axis=1)].sort_values("Maximum-Sharpe weight (%)", ascending=False)

print("Individual stock statistics:")
display(ind_stats.round(4))
print("Optimized portfolio statistics:")
display(port_stats_tab.round(4))
print("Optimized portfolio weights (only stocks with weight > 0.01%):")
display(weights.round(2))

## Section 4. Settings, inputs ($\mu$, $\Sigma$) and capped Markowitz solutions

### 4.1 Shared settings and inputs

**What the next cell does**

* defines the settings used by all optimisation sections: analysis period, 252 trading days, risk-free rate `RF = 0`, weight cap `MAX_W = 30 %` (used only for the efficient-frontier curve and for $\lambda$), number of annealing chains and sweeps of the own simulated annealer, random seed, optional Ledoit-Wolf shrinkage of the covariance matrix (`SHRINK_COV`, off) and the x-axis of the charts (variance);
* takes the prices from `adj_close` (Section 1), computes the daily simple returns, **keeps only the days on which every stock traded**, and computes once the annualised mean returns $\mu$ and covariance $\Sigma$ that **every** optimiser below receives;
* checks that $\Sigma$ is positive definite.

Nothing is forecast and there is no train/test split: the full period is used once.

In [ ]:
# SECTION 4.1 | Settings, data and inputs (mu, Sigma) - computed ONCE from the full period
# Uses the table `adj_close` of Section 1.  Every optimiser below receives exactly the same mu and Sigma.

# ---------------------------------- settings ----------------------------------
START, END  = "2000-01-01", "2026-01-01"           # full period used for the single optimisation
TD          = 252                                  # trading days per year
RF          = 0.0                                  # risk-free rate used in the optimisation sections
MAX_W       = 0.30                                 # cap per stock (used for the efficient-frontier curve and for lambda)
SA_CHAINS   = 128                                  # parallel annealing chains of the own simulated annealer
SA_SWEEPS   = 150                                  # sweeps per chain
SEED        = 42
SHRINK_COV  = False                                # True -> Ledoit-Wolf shrinkage of the covariance matrix
X_AXIS      = "variance"                           # "variance" (like the Markowitz chart) or "volatility"

# ------------------------------------ data ------------------------------------
prices = adj_close.sort_index().loc[START:END]
returns = prices.pct_change().iloc[1:]
n_rows = len(returns)
returns = returns.dropna()                          # only days on which every stock traded
tickers = list(returns.columns)
N = len(tickers)

def ledoit_wolf_cov(X):
    """Ledoit-Wolf shrinkage towards a scaled identity (same result as scikit-learn's LedoitWolf)."""
    X = X - X.mean(axis=0)
    T, n = X.shape
    S = X.T @ X / T
    m = np.trace(S) / n
    d2 = np.sum((S - m * np.eye(n)) ** 2)
    b2 = min((np.sum(np.sum(X ** 2, axis=1) ** 2) - T * np.sum(S ** 2)) / T ** 2, d2)
    sh = b2 / d2 if d2 > 0 else 0.0
    return (1 - sh) * S + sh * m * np.eye(n)

mu = returns.mean().to_numpy() * TD                                   # annualised mean return (arithmetic)
Sigma = (ledoit_wolf_cov(returns.to_numpy()) if SHRINK_COV else returns.cov().to_numpy()) * TD
assert np.all(np.linalg.eigvalsh(Sigma) > 0), "covariance matrix is not positive definite"

print(f"{N} stocks | {returns.index[0].date()} to {returns.index[-1].date()} | {len(returns)} daily returns "
      f"({n_rows - len(returns)} rows dropped for missing values)")
print(f"risk-free {RF:.1%} | weight cap {MAX_W:.0%} | covariance: {'Ledoit-Wolf' if SHRINK_COV else 'sample'}")

def xs(var):
    """x-axis coordinate of a portfolio from its variance."""
    var = np.asarray(var, dtype=float)
    return var if X_AXIS == "variance" else np.sqrt(var)
XLABEL = "Annualised portfolio variance" if X_AXIS == "variance" else "Annualised volatility"

### 4.2 Capped Markowitz solutions (reference portfolios)

The same mean-variance model as in Section 3, now with the settings of Section 4.1 and a **cap of 30 % per stock**. These are the classical reference the QUBO is compared with.

**What the next cell does**

* defines the helper functions: equal weight (EW), minimum variance, mean-variance $\min_w \tfrac{\lambda}{2}w^\top\Sigma w-\mu^\top w$ (exactly the objective the QUBO encodes), maximum Sharpe (tangency portfolio), the objective `mv_objective`, the statistics `port_stats` (return, variance, volatility, Sharpe) and the long-only efficient frontier `frontier_points`;
* solves them with SLSQP (long-only, fully invested, each weight at most `MAX_W`);
* computes the efficient frontier (`f_var`, `f_ret`) used in the later charts.

In [ ]:
# SECTION 4.2 | Classical Markowitz solutions (the reference the QUBO is compared with)
#   EW = equal weight | MinVar = minimum variance | MaxSharpe = tangency portfolio
#   MeanVar = minimise gamma/2 w'Sigma w - mu'w   (exactly the objective the QUBO encodes)
# All are long-only, fully invested, capped at MAX_W.

def equal_weight(n):
    return np.full(n, 1.0 / n)

def _slsqp(fun, jac, n, cap, starts):
    cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0, "jac": lambda w: np.ones(n)}]
    best = None
    for w0 in starts:
        res = minimize(fun, w0, jac=jac, method="SLSQP", bounds=[(0.0, cap)] * n,
                       constraints=cons, options={"maxiter": 1000, "ftol": 1e-12})
        if res.success and (best is None or res.fun < best.fun):
            best = res
    if best is None:
        return equal_weight(n)
    w = np.clip(best.x, 0.0, cap)
    return w / w.sum()

def min_variance(Sigma, cap):
    n = len(Sigma)
    return _slsqp(lambda w: w @ Sigma @ w, lambda w: 2 * Sigma @ w, n, cap, [equal_weight(n)])

def mean_variance(mu, Sigma, gamma, cap):
    n = len(mu)
    return _slsqp(lambda w: 0.5 * gamma * w @ Sigma @ w - mu @ w, lambda w: gamma * Sigma @ w - mu,
                  n, cap, [equal_weight(n), min_variance(Sigma, cap)])

def max_sharpe(mu, Sigma, cap, rf=RF):
    n = len(mu)
    def fun(w):
        return -(mu @ w - rf) / np.sqrt(w @ Sigma @ w)
    def jac(w):
        s = np.sqrt(w @ Sigma @ w)
        return -(mu / s - (mu @ w - rf) * (Sigma @ w) / s ** 3)
    return _slsqp(fun, jac, n, cap, [equal_weight(n), min_variance(Sigma, cap)])

def mv_objective(w, mu, Sigma, gamma):
    return 0.5 * gamma * w @ Sigma @ w - mu @ w

def port_stats(w, mu=mu, Sigma=Sigma):
    """(expected return, variance, volatility, Sharpe)"""
    ret, var = float(mu @ w), float(w @ Sigma @ w)
    return ret, var, np.sqrt(var), (ret - RF) / np.sqrt(var)

def frontier_points(mu, Sigma, cap, n_pts=80):
    """Long-only efficient frontier. Returns arrays (variance, return)."""
    n = len(mu)
    w_min = min_variance(Sigma, cap)
    order, w_hi, left = np.argsort(-mu), np.zeros(n), 1.0
    for i in order:                                    # highest return reachable under the cap
        take = min(cap, left); w_hi[i] = take; left -= take
        if left <= 1e-12: break
    pts = []
    for tr in np.linspace(mu @ w_min, mu @ w_hi, n_pts)[:-1]:
        cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0, "jac": lambda w: np.ones(n)},
                {"type": "eq", "fun": lambda w, tr=tr: mu @ w - tr, "jac": lambda w: mu}]
        res = minimize(lambda w: w @ Sigma @ w, w_min, jac=lambda w: 2 * Sigma @ w, method="SLSQP",
                       bounds=[(0.0, cap)] * n, constraints=cons, options={"maxiter": 1000, "ftol": 1e-12})
        if res.success:
            pts.append((res.fun, tr))
    pts.append((w_hi @ Sigma @ w_hi, mu @ w_hi))
    pts = np.array(pts)
    pts = pts[np.argsort(pts[:, 0])]
    return pts[:, 0], np.maximum.accumulate(pts[:, 1])      # variance ascending, return non-decreasing (efficient part only)

t0 = time.time()
w_ew = equal_weight(N)
w_minvar = min_variance(Sigma, MAX_W)
w_maxsh = max_sharpe(mu, Sigma, MAX_W)
t_cls = time.time() - t0
f_var, f_ret = frontier_points(mu, Sigma, MAX_W)

## Section 5. QUBO formulation

### 5.1 The model

Select exactly $k$ of the $N=20$ stocks and give each selected stock the weight $1/k$. With binary variables $x_i\in\{0,1\}$ ($x_i=1$ if stock $i$ is held), $w=x/k$ and the objective of the mean-variance model

$$f_k(x)=\frac{\lambda}{2}\,\frac{x^\top\Sigma x}{k^2}-\frac{\mu^\top x}{k},$$

the constraint $\sum_i x_i=k$ is added as a penalty:

$$H(x)=f_k(x)+P\Big(\sum_i x_i-k\Big)^2 .$$

Because $x_i^2=x_i$, $\;(\sum_i x_i-k)^2=x^\top(\mathbf 1\mathbf 1^\top-2k\,I)x+k^2$, so $H(x)=x^\top Qx+\text{const}$ with

$$Q=\frac{\lambda}{2k^2}\,\Sigma-\operatorname{diag}\!\Big(\frac{\mu}{k}\Big)+P\,(\mathbf 1\mathbf 1^\top-2k\,I),\qquad \text{const}=P\,k^2 .$$

For every $x$ with exactly $k$ ones the penalty is zero and $H(x)=f_k(x)$ (the Markowitz objective of the equal-weight portfolio).

### 5.2 Risk aversion $\lambda$

$\lambda$ is **not a free number**. It is the value for which the continuous problem $\min_w \tfrac{\lambda}{2}w^\top\Sigma w-\mu^\top w$ (long-only, fully invested) has the **maximum-Sharpe portfolio** $w_t$ as its optimum. For the tangency portfolio $\Sigma w_t=\dfrac{\sigma_t^2}{R_t-\text{RF}}(\mu-\text{RF}\,\mathbf 1)$, so the first-order condition $\lambda\Sigma w-\mu\propto\mathbf 1$ holds for

$$\lambda=\frac{R_t-\text{RF}}{\sigma_t^{2}} .$$

It depends only on $\mu$ and $\Sigma$. The next cell computes it and checks numerically that the continuous mean-variance optimum at this $\lambda$ equals the maximum-Sharpe portfolio (the 30 % cap must not be binding for this identity).

In [ ]:
# SECTION 5.2 | Risk aversion lambda = (R_t - RF) / sigma_t^2 from the maximum-Sharpe portfolio of Section 4.2
r_t, v_t, _, sh_t = port_stats(w_maxsh)
LAMBDA = (r_t - RF) / v_t
w_chk = mean_variance(mu, Sigma, LAMBDA, MAX_W)
print(f"lambda = (R_t - RF) / var_t = ({r_t:.4f} - {RF:.4f}) / {v_t:.5f} = {LAMBDA:.4f}")
print(f"check : continuous mean-variance optimum at this lambda vs max-Sharpe portfolio -> "
      f"max |weight difference| = {np.abs(w_chk - w_maxsh).max():.2e}  (largest weight {w_maxsh.max():.1%}, cap {MAX_W:.0%} not binding: {w_maxsh.max() < MAX_W - 1e-6})\n")

# objective of an (equal-weight or free-weight) portfolio, used by every solver below
obj_w = lambda w: 0.5 * LAMBDA * float(w @ Sigma @ w) - float(mu @ w)

### 5.3 Energy landscape: every feasible QUBO state in Markowitz space (illustration)

**What the next cell does**

For a small illustration with **its own parameters** ($\gamma=4$ as the risk-aversion, $K=8$ stocks, penalty $P=2\max\!\big(\max|\mu|/K,\;\tfrac{\gamma}{2}\max\Sigma_{ii}/K^2\big)$; these are independent of the $\lambda$ and $k$ used in the rest of the notebook) it

* enumerates **all** $\binom{20}{8}=125\,970$ feasible bit strings and computes for each the portfolio variance, expected return and QUBO energy;
* verifies with `assert` that the QUBO energy equals the Markowitz objective $\tfrac{\gamma}{2}w^\top\Sigma w-\mu^\top w$ on every feasible state;
* finds the exact QUBO ground state and plots all states in the (variance, return) plane together with the long-only Markowitz frontier (any weights), the best possible frontier with exactly $K$ equal-weight stocks and the iso-energy line (slope $\gamma/2$) through the QUBO optimum.

The code is wrapped in a function so that its variables do not overwrite the main ones.

In [ ]:
# SECTION 5.3 | QUBO energy landscape vs Markowitz frontier (illustration with its own gamma = 4, K = 8)
def qubo_landscape():
    """QUBO (select-K) energy landscape vs Markowitz frontier, with its own illustration parameters."""
    START, END = "2000-01-01", "2026-01-01"
    TD, GAMMA, K, PEN_MULT = 252, 4.0, 8, 2.0

    ret = adj_close.sort_index().loc[START:END].pct_change().iloc[1:].dropna()
    tick, N = list(ret.columns), ret.shape[1]
    mu, Sigma = ret.mean().to_numpy() * TD, ret.cov().to_numpy() * TD

    # QUBO exactly as in the notebook: f(z) = gamma/2 * w'Sigma w - mu'w + P (sum z - K)^2, with w = z/K
    P = PEN_MULT * max(np.abs(mu).max() / K, 0.5 * GAMMA * np.diag(Sigma).max() / K**2)
    Q = 0.5 * GAMMA / K**2 * Sigma - np.diag(mu / K) + P * (np.ones((N, N)) - 2 * K * np.eye(N))
    const = P * K**2

    # every one of the C(N,K) feasible bit strings
    idx = np.array(list(itertools.combinations(range(N), K)))
    Z = np.zeros((len(idx), N)); Z[np.arange(len(idx))[:, None], idx] = 1
    W = Z / K
    var, er = np.einsum("ij,jk,ik->i", W, Sigma, W), W @ mu
    E = np.einsum("ij,jk,ik->i", Z, Q, Z) + const                 # QUBO energy
    obj = 0.5 * GAMMA * var - er                                  # mean-variance objective
    assert np.allclose(E, obj, atol=1e-9), "QUBO energy != Markowitz objective"   # penalty is 0 when sum z = K
    g = np.argmin(E)                                              # exact QUBO ground state

    # continuous long-only Markowitz frontier (SLSQP)
    def frontier(targets):
        out, w0 = [], np.full(N, 1 / N)
        for t in targets:
            r = minimize(lambda w: w @ Sigma @ w, w0, jac=lambda w: 2 * Sigma @ w, method="SLSQP", bounds=[(0, 1)] * N,
                         constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}, {"type": "eq", "fun": lambda w, t=t: w @ mu - t}])
            if r.success: out.append((r.fun, t)); w0 = r.x
        return np.array(out)
    F = frontier(np.linspace(mu.min() * 1.02, mu.max() * 0.98, 80)); F = F[F[:, 1] >= F[np.argmin(F[:, 0]), 1]]   # efficient branch only

    # cardinality-K frontier = upper-left envelope of the subset cloud
    o = np.argsort(var); best, env = -np.inf, []
    for i in o:
        if er[i] > best: best = er[i]; env.append(i)

    fig, ax = plt.subplots(figsize=(10, 7))
    sc = ax.scatter(var, er, c=E, s=3, cmap="viridis_r", rasterized=True)
    fig.colorbar(sc, label=f"QUBO energy = γ/2·σ² − μ·w   (γ = {GAMMA:g})")
    ax.plot(F[:, 0], F[:, 1], "k-", lw=2, label="Markowitz frontier (long-only, any weights)")
    ax.plot(var[env], er[env], "r-", lw=2, label=f"Best possible with exactly K = {K} stocks")
    xs = np.array([var.min(), var.max()])
    ax.plot(xs, 0.5 * GAMMA * xs - E[g], "--", color="orange", lw=1.5, label="Iso-energy line through QUBO optimum (slope γ/2)")
    ax.scatter(var[g], er[g], marker="*", s=350, c="orange", edgecolor="k", zorder=5,
               label="QUBO ground state: " + ", ".join(np.array(tick)[idx[g]]))
    ax.set(xlabel="Portfolio variance  wᵀΣw", ylabel="Expected return  μᵀw",
           title=f"All {len(idx):,} feasible QUBO states (K = {K} of {N} stocks) in Markowitz mean-variance space")
    ax.legend(loc="lower right", fontsize=9); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/qubo_landscape.png", dpi=200)
    print(f"{len(idx):,} states | energy == objective verified | ground state: {np.array(tick)[idx[g]].tolist()} | E = {E[g]:.4f}")

qubo_landscape()

### 5.4 Penalty coefficient $P$ by Monte Carlo, for every $k=1\ldots 20$

The penalty must be large enough that breaking the constraint never pays, but not larger than necessary (a very large $P$ hides the financial terms of $Q$). It is estimated with Algorithm 1 of the paper (Eq. 8), with $A=(1,\ldots,1)$ and $b=k$:

1. draw `MC_N` random binary vectors $x$ (the number of ones is uniform on $0\ldots N$, positions random);
2. $x_{\text{from}}$ = the sampled vector with the smallest constraint violation $|\mathbf 1^\top x-k|$ (ties: the one with the lowest $f_k$);
3. $x_{\text{to}}$ = every sampled vector with $f_k(x_{\text{to}})<f_k(x_{\text{from}})$ and a larger violation (it "gains" by breaking the constraint);
4. candidate $P=\dfrac{f_k(x_{\text{from}})-f_k(x_{\text{to}})}{(\mathbf 1^\top x_{\text{to}}-k)^2-(\mathbf 1^\top x_{\text{from}}-k)^2}$, and the estimate is the **largest** candidate (a lower bound for $P$);
5. the procedure is repeated `MC_RUNS` times; the **largest estimate over all runs** is the penalty $P_k$ used in the QUBO.

If no sampled vector gains by breaking the constraint, the estimate of that run is $P=0$.

**What the next cell does**: runs this estimate for every $k=1\ldots 20$, prints the table (mean, standard deviation, minimum, median, maximum = $P$ used, number of runs with $P=0$), and draws a box plot of the estimates against $k$. The result `P_K` is used in all later sections.

In [ ]:
# SECTION 5.4 | Monte Carlo estimate of the QUBO penalty coefficient P for k = 1 ... 20
#
# QUBO (select exactly k of the N stocks, equal weights 1/k, full-period mu and Sigma from the previous cell):
#     H(x) = f_k(x) + P * (sum(x) - k)^2 ,       f_k(x) = lambda/2 * (x'Sigma x)/k^2 - (mu'x)/k
# Penalty estimate = Algorithm 1 of the paper (Eq. 8) with  A = (1,...,1),  b = k :
#   1. Monte Carlo: draw MC_N random binary vectors x                        -> x_set
#   2. x_from = vector in x_set with the smallest constraint violation |A x - b|
#              (ties -> the one with the lowest f_k, the strongest feasible competitor)
#   3. x_to   = every vector in x_set with  f_k(x_to) < f_k(x_from)           (it "gains" by breaking the constraint)
#   4. candidate P = ( f_k(x_from) - f_k(x_to) ) / ( (A x_to - b)^2 - (A x_from - b)^2 )
#   5. lower bound of P = the largest candidate
# The whole procedure is repeated MC_RUNS times with new random samples -> spread of the estimate for every k.

# settings
K_LIST  = list(range(1, N + 1))        # k = 1 ... 20  (N = 20 stocks)
MC_N    = 5000                         # random binary vectors per Monte Carlo run
MC_RUNS = 200                          # independent repetitions (gives the spread shown in the chart)
MC_SEED = SEED + 7

# Monte Carlo
def mc_binary_samples(n, m, rng):
    """m random binary vectors; the number of ones of each vector is uniform on 0..n, the positions are random."""
    card = rng.integers(0, n + 1, size=m)
    rank = np.argsort(np.argsort(rng.random((m, n)), axis=1), axis=1)
    return (rank < card[:, None]).astype(float)

def penalty_lower_bound(q, lin, s, k, lam):
    """Algorithm 1 for one sample set. q = x'Sigma x, lin = mu'x, s = sum(x) for every sampled vector."""
    f = 0.5 * lam / k ** 2 * q - lin / k             # x'Hx  (objective part of the QUBO)
    viol = (s - k) ** 2                              # (A x - b)^2
    v_min = viol.min()
    f_from = f[viol == v_min].min()                  # x_from
    m = (f < f_from) & (viol > v_min)                # x_to  (positive denominator)
    if not m.any():
        return 0.0          # no sampled vector gains by breaking the constraint -> this run needs no penalty (P = 0)
    return ((f_from - f[m]) / (viol[m] - v_min)).max()          # Eq. 8, largest candidate

rng_mc = np.random.default_rng(MC_SEED)
MC_P = np.zeros((MC_RUNS, len(K_LIST)))
t0 = time.time()
for run in range(MC_RUNS):
    Xs = mc_binary_samples(N, MC_N, rng_mc)
    q, lin, s = np.einsum("ri,ij,rj->r", Xs, Sigma, Xs), Xs @ mu, Xs.sum(axis=1)
    for j, k in enumerate(K_LIST):
        MC_P[run, j] = penalty_lower_bound(q, lin, s, k, LAMBDA)
print(f"Monte Carlo finished in {time.time() - t0:.1f} s  ({MC_RUNS} runs x {MC_N} random binary vectors, k = {K_LIST[0]}..{K_LIST[-1]})")

P_K = MC_P.max(axis=0)                 # largest lower bound found for every k  -> penalty used by the next cell
no_gain = [k for k, p in zip(K_LIST, P_K) if p == 0]
if no_gain:
    print(f"k = {no_gain}: in none of the {MC_RUNS} runs did any sampled vector gain by breaking the constraint, "
          f"so the estimated penalty is P = 0 for these k (the constraint is never attractive to break).\n")

# table
P_TABLE = pd.DataFrame({
    "k": K_LIST,
    "P mean": MC_P.mean(axis=0), "P std": MC_P.std(axis=0, ddof=1),
    "P min": MC_P.min(axis=0), "P median": np.median(MC_P, axis=0),
    "P max (used)": P_K,
    "runs with P = 0 (no violation gains)": (MC_P == 0).sum(axis=0),
}).set_index("k")
display(P_TABLE.round(5))
P_TABLE.to_csv(f"{OUT_DIR}/mc_penalty_vs_k.csv")

# chart: estimated penalty P versus k
fig, ax = plt.subplots(figsize=(14, 7))
data = [MC_P[:, j] for j in range(len(K_LIST))]
bp = ax.boxplot(data, positions=K_LIST, widths=0.6, patch_artist=True, showfliers=True,
                boxprops=dict(facecolor="#9ecae1", edgecolor="#08519c"), medianprops=dict(color="#d62728", lw=2),
                flierprops=dict(marker="o", ms=3, markerfacecolor="grey", markeredgecolor="grey", alpha=0.6))
ax.plot(K_LIST, MC_P.mean(axis=0), color="black", marker="o", ms=4, lw=1.2, label="Mean of the estimates")
ax.plot(K_LIST, P_K, color="#2ca02c", marker="^", ms=6, lw=1.2, ls="--", label="Largest estimate over all runs (penalty used for the QUBO)")
ax.set_xticks(K_LIST)
ax.set_xlabel("k  (number of stocks selected)"); ax.set_ylabel("Estimated penalty coefficient P (lower bound, Eq. 8)")
ax.grid(True, alpha=0.3); ax.legend(loc="best")
ax.set_title(f"Monte Carlo estimate of the QUBO penalty P versus k  ({MC_RUNS} runs x {MC_N} random binary vectors, "
             f"lambda = {LAMBDA:.3f})\nbox = quartiles, red line = median, whiskers/points = spread over repeated runs",
             fontsize=13, fontweight="bold")
plt.tight_layout(); savefig("mc_penalty_vs_k"); plt.show()

## Section 6. Classical solver 1: simulated annealing on the QUBO

### 6.1 The own simulated annealer and its helper functions

Simulated annealing minimises $x^\top Qx$ over $x\in\{0,1\}^N$. A move is accepted with the Metropolis rule: always if the energy change $\Delta E\le 0$, otherwise with probability $e^{-\Delta E/T}$. The temperature $T$ falls geometrically from $T_{\text{hi}}$ to $T_{\text{lo}}=10^{-4}\,T_{\text{hi}}$ over the sweeps, with $T_{\text{hi}}=0.1\times$ the median $|\Delta E|$ of single flips of random states.

**What the next cell defines**

* `simulated_annealing`: many independent chains in parallel. Each sweep proposes $N$ **single-bit flips** and $N$ **two-bit flips** (a two-bit flip of one selected and one unselected stock swaps a stock in for a stock out, so the number of selected stocks stays $k$);
* `repair_units`: if a chain ends with a broken budget, it adds or removes units greedily, always taking the best objective;
* `decode_select`: among the 10 lowest-energy chains it returns the best equal-weight portfolio (after repair if needed) and the share of chains that already hold exactly $k$ stocks;
* `chain_objectives_s`: the objective of every chain's portfolio (for distribution plots);
* `exact_select`: **exhaustive search** over all $\binom{N}{k}$ equal-weight portfolios, i.e. the exact optimum of the QUBO.

In [ ]:
# SECTION 6.1 | QUBO solver - simulated annealing (pure NumPy, no extra packages)
#   simulated_annealing : many parallel chains, single-bit and two-bit flips
#   repair_units        : if a chain ends with a slightly broken budget, add / remove units greedily
#   decode_select       : bits -> weights
#   exact_select        : exhaustive search over all C(N,K) portfolios (exact optimum of QUBO-S)

def simulated_annealing(Q, n_chains=128, n_sweeps=150, rng=None, t_hi=None, t_lo=None, t_scale=0.1):
    """Minimise x'Qx over x in {0,1}^n. Returns final states (chains x n), energies (chains), (temperatures, mean energy per sweep).
    Each sweep proposes n single-bit flips and n two-bit flips (the two-bit moves swap one stock / weight unit for another)."""
    rng = rng if rng is not None else np.random.default_rng(SEED)
    Q = 0.5 * (Q + Q.T)
    n = Q.shape[0]
    d = np.diag(Q).copy()
    off = Q - np.diag(d)
    X = rng.integers(0, 2, size=(n_chains, n)).astype(float)
    H = X @ off
    E = np.einsum("ri,ri->r", X @ Q, X)
    if t_hi is None:                                   # start temperature: a fraction of the typical single-flip energy change
        t_hi = t_scale * np.median(np.abs((1 - 2 * X) * (d + 2 * H)))
    if t_lo is None:
        t_lo = t_hi * 1e-4
    temps = np.geomspace(t_hi, t_lo, n_sweeps)
    trace = np.empty(n_sweeps)
    for s, T in enumerate(temps):
        U1 = rng.random((n, n_chains))
        for j, i in enumerate(rng.permutation(n)):                      # single-bit flips
            delta = 1 - 2 * X[:, i]
            dE = delta * (d[i] + 2 * H[:, i])
            acc = (dE <= 0) | (U1[j] < np.exp(-np.maximum(dE, 0) / T))
            if acc.any():
                m = acc * delta
                X[:, i] += m; H += m[:, None] * off[i]; E += acc * dE
        U2 = rng.random((n, n_chains))
        pairs = np.array([rng.choice(n, 2, replace=False) for _ in range(n)])
        for j, (i, k) in enumerate(pairs):                              # two-bit flips
            di, dk = 1 - 2 * X[:, i], 1 - 2 * X[:, k]
            dE = di * (d[i] + 2 * H[:, i]) + dk * (d[k] + 2 * H[:, k]) + 2 * off[i, k] * di * dk
            acc = (dE <= 0) | (U2[j] < np.exp(-np.maximum(dE, 0) / T))
            if acc.any():
                mi, mk = acc * di, acc * dk
                X[:, i] += mi; X[:, k] += mk
                H += mi[:, None] * off[i] + mk[:, None] * off[k]; E += acc * dE
        trace[s] = E.mean()
    return X, np.einsum("ri,ri->r", X @ Q, X), (temps, trace)

def repair_units(u, f, total, cap_u):
    """Greedy repair: add / remove single units until sum(u) == total, always taking the best objective."""
    u = u.copy()
    while u.sum() > total:
        cand = np.flatnonzero(u > 0)
        vals = []
        for i in cand:
            u[i] -= 1; vals.append(f(u)); u[i] += 1
        u[cand[int(np.argmin(vals))]] -= 1
    while u.sum() < total:
        cand = np.flatnonzero(u < cap_u)
        vals = []
        for i in cand:
            u[i] += 1; vals.append(f(u)); u[i] -= 1
        u[cand[int(np.argmin(vals))]] += 1
    return u

def decode_select(X, E, q, mu, Sigma, gamma, top=10):
    k = q["k"]
    f = lambda uu: mv_objective(uu / k, mu, Sigma, gamma)
    best_u, best_f = None, np.inf
    for r in np.argsort(E)[:top]:
        u = repair_units(X[r].astype(int), f, k, 1)
        if f(u) < best_f:
            best_u, best_f = u, f(u)
    return best_u / k, dict(feasible_rate=float((X.sum(axis=1) == k).mean()), units=best_u)

def chain_objectives_s(X, q, mu, Sigma, gamma):
    k = q["k"]
    f = lambda uu: mv_objective(uu / k, mu, Sigma, gamma)
    return np.array([f(repair_units(z.astype(int), f, k, 1)) for z in X])

def exact_select(mu, Sigma, gamma, k, chunk=40000):
    """Exhaustive search over all C(N,k) equal-weight portfolios. Returns (best index set, best objective)."""
    n = len(mu)
    best_f, best_c = np.inf, None
    it = itertools.combinations(range(n), k)
    while True:
        block = list(itertools.islice(it, chunk))
        if not block:
            break
        c = np.array(block)
        var = Sigma[c[:, :, None], c[:, None, :]].sum(axis=(1, 2)) / k ** 2
        f = 0.5 * gamma * var - mu[c].sum(axis=1) / k
        j = int(np.argmin(f))
        if f[j] < best_f:
            best_f, best_c = float(f[j]), c[j]
    return best_c, best_f

### 6.2 QUBO solved for every $k=1\ldots 20$ (simulated annealing + exact check)

**What the next cell does**

For each $k$ it builds $Q$ with the penalty $P_k$ of Section 5.4, anneals `SA_CHAINS` chains for `SA_SWEEPS` sweeps, decodes the best portfolio, checks that the QUBO energy of the chosen bit string equals the Markowitz objective (no penalty is active), and compares with the exhaustive optimum of `exact_select`. It prints the portfolio of every $k$, a table (return, variance, volatility, Sharpe, objective of the annealer and of the exact search, whether the annealer found the exact optimum, share of feasible chains, share of chains at the exact optimum, $P$ used), saves the tables as CSV, and draws random equal-weight $k$-stock portfolios (dots, up to 3000 per $k$) with the efficient frontier and the QUBO solution of every $k$.

In [ ]:
# SECTION 6.2 | QUBO solution for k = 1 ... 20  (simulated annealing + exact check)
# Needs the previous new cell (LAMBDA and the penalty P_K from the Monte Carlo) and the earlier cell's functions.
#     H(x) = lambda/2 * (x'Sigma x)/k^2 - (mu'x)/k + P_k * (sum(x) - k)^2 ,  x_i in {0,1}, weight of a chosen stock = 1/k
#     x'Qx + const  with  Q = lambda/(2k^2) Sigma - diag(mu/k) + P_k (11' - 2k I),  const = P_k k^2

N_RANDOM = 3000                       # random k-stock portfolios drawn for each k (all of them if C(N,k) <= N_RANDOM)
QK_SEED  = SEED + 11

def build_qubo_select_P(mu, Sigma, lam, k, P):
    n = len(mu)
    Q = 0.5 * lam / k ** 2 * Sigma - np.diag(mu / k) + P * (np.ones((n, n)) - 2 * k * np.eye(n))
    return dict(Q=Q, const=P * k ** 2, n=n, k=k, P=P)

def random_subsets(n, k, m, rng):
    from itertools import combinations
    if comb(n, k) <= m:
        return np.array(list(combinations(range(n), k)))
    return np.argsort(rng.random((m, n)), axis=1)[:, :k]

rows, SOL = [], {}
rng_k = np.random.default_rng(QK_SEED)
t0 = time.time()
for j, k in enumerate(K_LIST):
    q = build_qubo_select_P(mu, Sigma, LAMBDA, k, P_K[j])
    X, E, _ = simulated_annealing(q["Q"], SA_CHAINS, SA_SWEEPS, rng=np.random.default_rng(SEED + 1000 + k))
    w, info = decode_select(X, E, q, mu, Sigma, LAMBDA)
    z = (w > 0).astype(float)
    # the QUBO energy of the chosen bit string must equal the objective (no penalty, the constraint holds)
    assert z.sum() == k and np.isclose(z @ q["Q"] @ z + q["const"], mv_objective(w, mu, Sigma, LAMBDA))
    sel_ex, f_ex = exact_select(mu, Sigma, LAMBDA, k)
    f_sa = mv_objective(w, mu, Sigma, LAMBDA)
    chain_f = chain_objectives_s(X, q, mu, Sigma, LAMBDA)
    ret, var, vol, sh = port_stats(w)
    SOL[k] = w
    rows.append({"k": k, "Selected stocks": ", ".join(t for t, wi in zip(tickers, w) if wi > 0),
                 "Weight each (%)": 100 / k, "Exp. return (%)": ret * 100, "Variance": var, "Volatility (%)": vol * 100,
                 "Sharpe": sh, "Objective (QUBO/SA)": f_sa, "Objective (exact)": f_ex,
                 "SA = exact optimum": bool(f_sa <= f_ex + 1e-10 and set(np.flatnonzero(w)) == set(sel_ex)),
                 "Chains feasible (%)": info["feasible_rate"] * 100,
                 "Chains at exact optimum (%)": float(np.mean(chain_f <= f_ex + 1e-10)) * 100,
                 "P used": P_K[j]})
print(f"Solved {len(K_LIST)} QUBOs (annealing {SA_CHAINS} chains x {SA_SWEEPS} sweeps + exhaustive check) in {time.time() - t0:.1f} s\n")

QK = pd.DataFrame(rows).set_index("k")
for k, r in QK.iterrows():
    print(f"k = {k:2d} | {r['Selected stocks']}  (each {r['Weight each (%)']:.2f}%)  |  return {r['Exp. return (%)']:.2f}%, "
          f"variance {r['Variance']:.4f}, volatility {r['Volatility (%)']:.2f}%, Sharpe {r['Sharpe']:.3f}  |  "
          f"matches exact optimum: {r['SA = exact optimum']}")
print()
display(QK.drop(columns=["Selected stocks"]).round(4))
QK.to_csv(f"{OUT_DIR}/qubo_solution_k_1_to_20.csv")
pd.DataFrame(SOL, index=tickers).T.mul(100).round(4).rename_axis("k").to_csv(f"{OUT_DIR}/qubo_weights_k_1_to_20.csv")

# chart: random k-stock portfolios and the QUBO solution of every k
fig, ax = plt.subplots(figsize=(14, 8.5))
cmap = plt.get_cmap("turbo"); norm = plt.Normalize(K_LIST[0], K_LIST[-1])
for k in K_LIST:
    idx = random_subsets(N, k, N_RANDOM, rng_k)
    r_ = mu[idx].mean(axis=1)
    v_ = Sigma[idx[:, :, None], idx[:, None, :]].sum(axis=(1, 2)) / k ** 2
    ax.scatter(xs(v_), r_, s=6, alpha=0.18, color=cmap(norm(k)), linewidths=0, rasterized=True)
ax.plot(xs(f_var), f_ret, color="red", lw=2.5, label=f"Efficient frontier (previous cell, long-only, cap {MAX_W:.0%})", zorder=4)
qv = np.array([port_stats(SOL[k])[1] for k in K_LIST]); qr = np.array([port_stats(SOL[k])[0] for k in K_LIST])
ax.plot(xs(qv), qr, color="black", lw=1, ls="--", zorder=5)
ax.scatter(xs(qv), qr, c=K_LIST, cmap=cmap, norm=norm, marker="^", s=170, edgecolor="black", linewidths=1.2, zorder=6,
           label="QUBO solution for each k (simulated annealing)")
for k, x_, y_ in zip(K_LIST, xs(qv), qr):
    ax.annotate(f"k={k}", (x_, y_), xytext=(7, 6), textcoords="offset points", fontsize=9, fontweight="bold", zorder=7)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
cb = plt.colorbar(sm, ax=ax, pad=0.01); cb.set_label("k (number of stocks)")
ax.set_xlabel(XLABEL); ax.set_ylabel("Annualised expected return"); ax.grid(True, alpha=0.3); ax.legend(loc="lower right")
ax.set_title(f"Random equal-weight k-stock portfolios (dots, up to {N_RANDOM} per k) and the QUBO solution for k = 1 ... {N}\n"
             f"full-period {returns.index[0].year}-{returns.index[-1].year} mu and Sigma, lambda = {LAMBDA:.3f}",
             fontsize=13, fontweight="bold")
plt.tight_layout(); savefig("qubo_k_solutions_random_portfolios"); plt.show()

### 6.3 Choice of $k$ and the selected portfolio

**How $k$ is chosen**: the QUBO is solved for every $k=1\ldots 20$ (Section 6.2); the chosen $k^\star$ is the one whose QUBO solution has the **lowest value of the same risk-return objective** $\tfrac{\lambda}{2}w^\top\Sigma w-\mu^\top w$. From here on `K = K_STAR` in all solvers.

**What the next cell does**

* finds $k^\star$ and prints the selected stocks (equal weight $1/k^\star$ each) with their stand-alone return, volatility and Sharpe ratio;
* compares the QUBO selection with equal weight over all 20 stocks, the Markowitz maximum-Sharpe and minimum-variance portfolios (same $\mu$, $\Sigma$, $\lambda$ for every row);
* Chart 1: risk-return map of the 20 stocks with the efficient frontier, the QUBO solution of every $k$ and the choice $k^\star$, next to the objective and Sharpe ratio as functions of $k$;
* Chart 2: stand-alone Sharpe ratio of every stock, selected stocks in green.

In [ ]:
# SECTION 6.3 | choice of k and the selected portfolio
#  choose k and report 
K_STAR = int(QK["Objective (QUBO/SA)"].idxmin())
w_star = SOL[K_STAR]
sel = np.flatnonzero(w_star)
print("=" * 78)
print(f"QUBO RESULT: choose k = {K_STAR} of the {N} stocks, equal weight {100 / K_STAR:.2f}% each")
print("=" * 78)
stock_tab = pd.DataFrame({"Annualised return (%)": mu * 100, "Volatility (%)": np.sqrt(np.diag(Sigma)) * 100,
                          "Weight (%)": w_star * 100}, index=tickers)
stock_tab["Sharpe (stand-alone)"] = (mu - RF) / np.sqrt(np.diag(Sigma))
stock_tab["Selected"] = np.where(w_star > 0, "YES", "-")
display(stock_tab.sort_values(["Selected", "Weight (%)"], ascending=[False, False]).round(3))

cmp_rows = {}
for name, w_ in [(f"QUBO selection (k = {K_STAR})", w_star), ("Equal weight, all 20 stocks", w_ew),
                 ("Markowitz max-Sharpe", w_maxsh), ("Markowitz min-variance", w_minvar)]:
    r_, v_, s_, h_ = port_stats(w_)
    cmp_rows[name] = {"Exp. return (%)": r_ * 100, "Variance": v_, "Volatility (%)": s_ * 100, "Sharpe": h_,
                      "Objective": mv_objective(w_, mu, Sigma, LAMBDA), "No. of stocks": int((w_ > 1e-6).sum())}
CMP = pd.DataFrame(cmp_rows).T
print("\nFull-period risk / return of the QUBO selection (same mu and Sigma for every row):")
display(CMP.round(4))
CMP.to_csv(f"{OUT_DIR}/qubo_selection_comparison.csv")
stock_tab.to_csv(f"{OUT_DIR}/qubo_selection_stocks.csv")

#  chart 1: risk-return map + choice of k 
sig2 = np.diag(Sigma)
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
ax = axes[0]
ax.plot(xs(f_var), f_ret, color="red", lw=2.5, label=f"Efficient frontier (long-only, cap {MAX_W:.0%})", zorder=2)
ax.scatter(xs(sig2[w_star == 0]), mu[w_star == 0], s=70, color="lightgrey", edgecolor="grey", label="Stocks not selected", zorder=3)
ax.scatter(xs(sig2[sel]), mu[sel], s=110, color="#2ca02c", edgecolor="black", label="Stocks selected by QUBO", zorder=4)
for i, t in enumerate(tickers):
    ax.annotate(t, (xs(sig2[i]), mu[i]), xytext=(5, 4), textcoords="offset points", fontsize=8)
qv = np.array([port_stats(SOL[k])[1] for k in K_LIST]); qr = np.array([port_stats(SOL[k])[0] for k in K_LIST])
ax.plot(xs(qv), qr, color="black", lw=1, ls="--", marker="^", ms=6, label="QUBO solution for k = 1..20", zorder=5)
rs, vs_, _, _ = port_stats(w_star)
ax.scatter(xs(vs_), rs, marker="*", s=600, color="gold", edgecolor="black", linewidths=1.5, zorder=6,
           label=f"QUBO choice: k = {K_STAR} stocks")
ax.set_xlabel(XLABEL); ax.set_ylabel("Annualised expected return"); ax.grid(True, alpha=0.3); ax.legend(loc="lower right")
ax.set_title("Risk-return map of the 20 stocks and the QUBO selection", fontweight="bold")

ax = axes[1]
ax.plot(K_LIST, QK["Objective (QUBO/SA)"], color="#1f77b4", marker="o", lw=2, label="Objective  lambda/2 w'Sigma w - mu'w  (lower = better)")
ax.axvline(K_STAR, color="gold", lw=3, alpha=0.8, label=f"chosen k = {K_STAR}")
ax.set_xlabel("k  (number of stocks selected)"); ax.set_ylabel("Objective of the QUBO solution", color="#1f77b4")
ax.set_xticks(K_LIST); ax.grid(True, alpha=0.3)
ax2 = ax.twinx(); ax2.plot(K_LIST, QK["Sharpe"], color="#ff7f0e", marker="s", lw=2, label="Sharpe ratio"); ax2.set_ylabel("Sharpe ratio", color="#ff7f0e")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax.legend(h1 + h2, l1 + l2, loc="best")
ax.set_title("Quality of the QUBO solution for each k", fontweight="bold")
fig.suptitle(f"QUBO portfolio selection on {N} US stocks, {returns.index[0].year}-{returns.index[-1].year}  (lambda = {LAMBDA:.3f})",
             fontsize=15, fontweight="bold")
plt.tight_layout(); savefig("qubo_selection_riskreturn_and_k"); plt.show()

#  chart 2: the selected stocks 
order = np.argsort(-(mu - RF) / np.sqrt(sig2))
fig, ax = plt.subplots(figsize=(14, 6.5))
colors = ["#2ca02c" if w_star[i] > 0 else "lightgrey" for i in order]
ax.bar([tickers[i] for i in order], ((mu - RF) / np.sqrt(sig2))[order], color=colors, edgecolor="black")
ax.set_ylabel("Stand-alone Sharpe ratio (full period)"); ax.grid(True, axis="y", alpha=0.3)
ax.set_title(f"Stocks chosen by QUBO (green, weight {100 / K_STAR:.2f}% each) versus the other stocks", fontweight="bold")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); savefig("qubo_selection_stocks"); plt.show()

K = K_STAR            # number of stocks used by every solver from here on

### 6.4 The same QUBO sampled with D-Wave's classical simulated-annealing sampler

The QUBO of Section 5.1 for $k=k^\star$ is written as a `dimod.BinaryQuadraticModel` and sampled with the `SimulatedAnnealingSampler` of `dwave-samplers` (classical, runs on your computer, no account needed). `SAMPLER = "qpu"` would send it to a real D-Wave quantum annealer instead (needs a Leap account and API token).

**What the next cell does**

* takes the penalty $P=P_{k^\star}$ of Section 5.4 (the same value the Monte Carlo gives for this $k$), builds $Q$ and the constant, and the `BinaryQuadraticModel`;
* samples `NUM_READS = 1000` reads with `NUM_SWEEPS = 1000` sweeps each (seed `SEED`);
* aggregates the reads, checks that every sampler energy equals $x^\top Qx+\text{const}$, and reports the share of reads that hold exactly $k$ stocks and the share that reach the exact optimum (found by exhaustive search over all $\binom{20}{k}$ equal-weight portfolios);
* prints the best feasible sample, the 10 lowest-energy distinct samples, and draws (left) the histogram of the sampled energies with the exact optimum and (right) the sampled portfolios in the risk-return plane.

In [ ]:
# SECTION 6.4 | D-WAVE (classical SA sampler) version of the k-stock QUBO, k = K*
#
# The same QUBO as before, written as a dimod BinaryQuadraticModel and sampled with an Ocean sampler:
#     H(x) = lambda/2 * (x'Sigma x)/k^2 - (mu'x)/k + P * (sum(x) - k)^2 ,     x_i in {0,1},  weight of a chosen stock = 1/k
#   mu, Sigma : full-period 2000-2025 annualised mean return / covariance (from the downloaded data above)
#   lambda    : (R_t - RF)/sigma_t^2  (the value for which the max-Sharpe portfolio is the continuous mean-variance optimum)
#   P         : Monte Carlo penalty estimate (Algorithm 1 of the paper, Eq. 8), largest estimate of all runs
#
#   SAMPLER = "local"  -> D-Wave's classical simulated-annealing sampler on your own computer (free, no account)
#   SAMPLER = "qpu"    -> the real D-Wave quantum annealer (needs a D-Wave Leap account and API token)
# The result is compared with the exhaustive optimum over all C(20,k) subsets.

#  settings 
SAMPLER    = "local"                   # "local" (simulated annealing, classical)  or  "qpu" (D-Wave quantum annealer)
NUM_READS  = 1000                      # number of samples (anneals) requested
NUM_SWEEPS = 1000                      # local sampler only: sweeps per read
ANNEAL_US  = None                      # qpu only: annealing time in microseconds (None = default 20)
CHAIN_STRENGTH = None                  # qpu only: None = automatic
DWAVE_TOKEN = os.environ.get("DWAVE_API_TOKEN")   # qpu only: your Leap API token (or run `dwave config create` once)

# penalty of Section 5.4 for this k (largest Monte Carlo estimate over all runs)
P = float(P_K[K - 1])
print(f"penalty P for k = {K}: Monte Carlo max over {MC_RUNS} runs = {P:.5f}")

# ----------------------------- QUBO -> dimod BinaryQuadraticModel ------------------
Q = 0.5 * LAMBDA / K ** 2 * Sigma - np.diag(mu / K) + P * (np.ones((N, N)) - 2 * K * np.eye(N))     # H = x'Qx + const
CONST = P * K ** 2
dimod = _ensure("dimod", "dimod")
bqm = dimod.BinaryQuadraticModel.from_qubo({(i, j): (Q[i, j] + Q[j, i] if i != j else Q[i, i])
                                            for i in range(N) for j in range(i, N)}, offset=CONST)
print(f"BQM: {bqm.num_variables} binary variables, {bqm.num_interactions} couplings (fully connected)")

# sample
t0 = time.time()
if SAMPLER == "local":
    sa = _ensure("dwave.samplers", "dwave-samplers")
    sampleset = sa.SimulatedAnnealingSampler().sample(bqm, num_reads=NUM_READS, num_sweeps=NUM_SWEEPS, seed=SEED)
    sampler_name = "D-Wave simulated annealing (classical, local)"
elif SAMPLER == "qpu":
    ds = _ensure("dwave.system", "dwave-system")
    qpu = ds.DWaveSampler(token=DWAVE_TOKEN) if DWAVE_TOKEN else ds.DWaveSampler()
    kw = dict(num_reads=NUM_READS, label=f"QUBO {K}-stock portfolio")
    if ANNEAL_US: kw["annealing_time"] = ANNEAL_US
    if CHAIN_STRENGTH: kw["chain_strength"] = CHAIN_STRENGTH
    sampleset = ds.EmbeddingComposite(qpu).sample(bqm, **kw)
    sampler_name = f"D-Wave quantum annealer ({qpu.properties.get('chip_id', 'QPU')})"
else:
    raise ValueError('SAMPLER must be "local" or "qpu"')
t_sample = time.time() - t0
print(f"Sampler: {sampler_name}  |  {NUM_READS} reads in {t_sample:.1f} s")
if SAMPLER == "qpu":
    cbf = sampleset.record.chain_break_fraction.mean() if "chain_break_fraction" in sampleset.record.dtype.names else float("nan")
    print(f"QPU access time {sampleset.info.get('timing', {}).get('qpu_access_time', float('nan')) / 1e3:.1f} ms | "
          f"mean chain-break fraction {cbf:.3f}")

# analyse the samples
ss = sampleset.aggregate()
cols = [list(ss.variables).index(i) for i in range(N)]
X = np.array(ss.record.sample)[:, cols].astype(float)
counts = np.array(ss.record.num_occurrences); E = np.array(ss.record.energy)
assert np.allclose(E, np.einsum("ui,ij,uj->u", X, Q, X) + CONST, atol=1e-8), "energy mismatch"
feas = X.sum(axis=1) == K
Wmat = X / K
objs = np.array([obj_w(w) if f else np.nan for w, f in zip(Wmat, feas)])      # risk-return objective of the equal-weight portfolio

def best_equal_weight(K, chunk=40000):
    best_f, best_c, it = np.inf, None, combinations(range(N), K)
    while True:
        c = np.array(list(islice(it, chunk)))
        if len(c) == 0:
            return best_f, best_c
        f = 0.5 * LAMBDA * Sigma[c[:, :, None], c[:, None, :]].sum(axis=(1, 2)) / K ** 2 - mu[c].sum(axis=1) / K
        if f.min() < best_f:
            best_f, best_c = float(f.min()), c[int(np.argmin(f))]
f_ex, c_ex = best_equal_weight(K)

best = int(np.nanargmin(objs)) if feas.any() else None
total = counts.sum()
print(f"\nReads that satisfy 'exactly {K} stocks': {counts[feas].sum() / total * 100:.1f}%   |   distinct solutions found: {len(counts)}")
if best is None:
    raise RuntimeError("No sampled solution has exactly k stocks: increase NUM_READS or P.")
hit = feas & np.isclose(objs, f_ex, atol=1e-10)
print(f"Reads that found the exact optimum: {counts[hit].sum() / total * 100:.1f}%")
DWAVE_QUBO_HIT = counts[hit].sum() / total * 100        # kept for the Ising comparison (Section 8)
sel = np.flatnonzero(X[best])
print("=" * 78)
print(f"SAMPLER RESULT (best feasible sample, k = {K}): " + ", ".join(tickers[i] for i in sel) + f"  -> {100 / K:.0f}% each")
print(f"exhaustive optimum                           : " + ", ".join(tickers[i] for i in sorted(c_ex)))
print(f"same stocks as exhaustive optimum: {set(sel) == set(c_ex)}   |   objective {objs[best]:.6f} vs exact {f_ex:.6f}")
print("=" * 78)

order = np.argsort(E)[:10]
top = pd.DataFrame({"Stocks": [", ".join(tickers[i] for i in np.flatnonzero(X[u])) for u in order],
                    "Reads": counts[order], "Energy": E[order], "Exactly k stocks": feas[order],
                    "Objective": objs[order], "Gap to exact optimum": objs[order] - f_ex})
print("The 10 lowest-energy distinct samples:")
display(top.round(6))
r_, v_, s_, h_ = port_stats(Wmat[best])
print(f"Best sample portfolio: return {r_ * 100:.2f}% | variance {v_:.4f} | volatility {s_ * 100:.2f}% | Sharpe {h_:.3f}")
top.to_csv(f"{OUT_DIR}/dwave_k{K}_top_samples.csv")

# charts 
allc = np.array(list(combinations(range(N), K))) if comb(N, K) <= 300000 else None
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
ax = axes[0]
bins = np.linspace(E.min(), np.percentile(np.repeat(E, counts), 99), 40)
ax.hist(np.repeat(E[feas], counts[feas]), bins=bins, color="#2ca02c", alpha=0.8, label=f"exactly {K} stocks (feasible)")
if (~feas).any():
    ax.hist(np.repeat(E[~feas], counts[~feas]), bins=bins, color="#d62728", alpha=0.6, label=f"wrong number of stocks")
e_opt = float(np.einsum("i,ij,j", (lambda z: z)(np.bincount(c_ex, minlength=N).astype(float)), Q,
                        np.bincount(c_ex, minlength=N).astype(float)) + CONST)
ax.axvline(e_opt, color="gold", lw=3, label="exact optimum")
ax.set_xlabel("QUBO energy of the sample"); ax.set_ylabel(f"number of reads (of {NUM_READS})"); ax.grid(True, alpha=0.3); ax.legend()
ax.set_title("Energies of the sampled solutions", fontweight="bold")

ax = axes[1]
if allc is not None:
    rc = mu[allc].mean(axis=1); vc = Sigma[allc[:, :, None], allc[:, None, :]].sum(axis=(1, 2)) / K ** 2
    ax.scatter(xs(vc), rc, s=8, color="#9ecae1", alpha=0.5, linewidths=0, label=f"All {len(allc)} possible {K}-stock portfolios")
ax.plot(xs(f_var), f_ret, color="red", lw=2.5, label=f"Efficient frontier (cap {MAX_W:.0%})")
fi = np.flatnonzero(feas)
sr = np.array([port_stats(Wmat[u])[0] for u in fi]); sv = np.array([port_stats(Wmat[u])[1] for u in fi])
ax.scatter(xs(sv), sr, s=40 + 260 * counts[fi] / counts[fi].max(), color="#ff7f0e", edgecolor="black", alpha=0.85, zorder=4,
           label="Distinct sampled portfolios (size = number of reads)")
ax.scatter(xs(v_), r_, marker="*", s=650, color="gold", edgecolor="black", linewidths=1.5, zorder=6, label="Best sampled portfolio")
ax.set_xlabel(XLABEL); ax.set_ylabel("Annualised expected return"); ax.grid(True, alpha=0.3); ax.legend(loc="lower right")
ax.set_title("Where the sampled portfolios lie", fontweight="bold")
fig.suptitle(f"{sampler_name}: QUBO selection of {K} of {N} stocks  (lambda = {LAMBDA:.3f}, P = {P:.4f})", fontsize=15, fontweight="bold")
plt.tight_layout(); savefig(f"dwave_k{K}_{SAMPLER}"); plt.show()

## Section 7. Classical solver 2: MIQP (mixed-integer quadratic programming)

The MIQP chooses **which** $k$ stocks to hold and their **optimal (not necessarily equal) weights** at the same time:

$$\min_{w,z}\;\frac{\lambda}{2}\,w^\top\Sigma w-\mu^\top w\quad\text{s.t.}\quad \sum_i w_i=1,\;\; \sum_i z_i=k,\;\; W_{\min}z_i\le w_i\le z_i,\;\; z_i\in\{0,1\}.$$

$z_i=1$ means stock $i$ may be held and $w_i$ is its continuous weight. With $W_{\min}=0$ this is exactly the model above. It is solved **exactly** by branch and bound (Gurobi; SCIP is used if Gurobi cannot be used). $\mu$, $\Sigma$, $\lambda$ and $k=k^\star$ are the same as for the QUBO, so the only difference to the QUBO is the weight model (free weights instead of $1/k$).

**What the next cell does**

* sets the minimum weight `W_MIN = 0` and the switch `ENUM_CHECK`, which enables an exhaustive verification of the MIQP result (solving the convex QP of every $k$-subset) when $\binom{N}{k}\le 6000$;
* replaces $\Sigma$ by $\Sigma_{\text{psd}}$ (negative rounding eigenvalues clipped) so the objective is convex;
* solves the MIQP, then computes the exact optimum of the equal-weight (QUBO-type) problem for comparison;
* prints the held stocks and weights, compares MIQP / equal-weight QUBO / equal weight of all 20 / maximum-Sharpe / minimum-variance, and draws the risk-return map and the weights.

In [ ]:
# SECTION 7 | MIQP (Mixed-Integer Quadratic Programming) for the k-stock Markowitz problem, k = K*
#
#     minimise    lambda/2 * w'Sigma w  -  mu'w
#     subject to  sum(w) = 1 ,   sum(z) = k ,   0 <= w_i <= z_i ,   z_i in {0,1}
#
#   z_i = 1  : stock i may be held        w_i : continuous portfolio weight (not forced to be equal)
#   mu, Sigma : full-period 2000-2025 annualised mean return / covariance (no train/test split, no backtest)
#   lambda    : the same risk aversion as in the QUBO code: the value for which the continuous mean-variance optimum is the
#               max-Sharpe portfolio,  lambda = (R_t - RF)/sigma_t^2  (from mu and Sigma only)
#   solver    : exact branch-and-bound by a MIQP solver (Gurobi or SCIP); the result is verified by solving the
#               convex QP of every possible k-stock subset (exhaustive check).
# Needs:  gurobipy (Gurobi; the pip licence is large enough) and/or pyscipopt (SCIP, free). Missing packages are installed automatically
#         into the Python that runs the notebook. No cvxpy needed.

W_MIN    = 0.0          # 0 -> exactly your model (0 <= w_i <= z_i). Set e.g. 0.05 to force every chosen stock to weigh >= 5%
ENUM_CHECK = comb(N, K) <= 6000      # exhaustive verification of the MIQP answer (4845 subsets for k = 4)

# MIQP model
ev, V = np.linalg.eigh(Sigma)
Sigma_psd = (V * np.clip(ev, 0, None)) @ V.T                    # Sigma without tiny negative rounding eigenvalues (convex objective)

def solve_gurobi():
    gp = _ensure("gurobipy", "gurobipy")
    from gurobipy import GRB
    m = gp.Model("miqp_k_stocks"); m.Params.OutputFlag = 0; m.Params.MIPGap = 1e-9; m.Params.MIPGapAbs = 1e-12
    w = m.addMVar(N, lb=0.0, ub=1.0, name="w"); z = m.addMVar(N, vtype=GRB.BINARY, name="z")
    m.setObjective(0.5 * LAMBDA * (w @ Sigma_psd @ w) - mu @ w, GRB.MINIMIZE)
    m.addConstr(w.sum() == 1); m.addConstr(z.sum() == K); m.addConstr(w <= z); m.addConstr(w >= W_MIN * z)
    m.optimize()
    if m.SolCount == 0:
        raise RuntimeError(f"Gurobi status {m.Status}")
    return w.X.copy(), z.X.copy(), f"status {m.Status} (2 = optimal), MIP gap {m.MIPGap:.1e}"

def solve_scip():
    sc = _ensure("pyscipopt", "pyscipopt")
    m = sc.Model("miqp_k_stocks"); m.hideOutput(); m.setParam("limits/gap", 0.0); m.setParam("limits/absgap", 0.0)
    w = [m.addVar(lb=0.0, ub=1.0, name=f"w{i}") for i in range(N)]
    z = [m.addVar(vtype="B", name=f"z{i}") for i in range(N)]
    t = m.addVar(lb=None, name="t")                                 # epigraph variable: t >= lambda/2 w'Sigma w - mu'w
    m.setObjective(t, "minimize")
    m.addCons(t >= 0.5 * LAMBDA * sc.quicksum(float(Sigma_psd[i, j]) * w[i] * w[j] for i in range(N) for j in range(N))
              - sc.quicksum(float(mu[i]) * w[i] for i in range(N)))
    m.addCons(sc.quicksum(w) == 1); m.addCons(sc.quicksum(z) == K)
    for i in range(N):
        m.addCons(w[i] <= z[i]); m.addCons(w[i] >= W_MIN * z[i])
    m.optimize()
    if m.getNSols() == 0:
        raise RuntimeError(f"SCIP status {m.getStatus()}")
    return (np.array([m.getVal(v) for v in w]), np.array([m.getVal(v) for v in z]),
            f"status {m.getStatus()}, gap {m.getGap():.1e}")

SOLVER, t0 = None, time.time()
for name, fn in [("Gurobi", solve_gurobi), ("SCIP", solve_scip)]:
    try:
        w_raw, z_raw, status = fn(); SOLVER = name; break
    except Exception as e:
        print(f"{name} could not be used: {e}")
if SOLVER is None:
    raise RuntimeError("No MIQP solver worked. In a notebook cell run:  %pip install gurobipy pyscipopt   then restart the kernel.")
t_miqp = time.time() - t0
w_miqp = np.where(w_raw < 1e-8, 0.0, w_raw); w_miqp /= w_miqp.sum()
z_miqp = np.flatnonzero(np.round(z_raw) > 0.5)
print(f"MIQP solved with {SOLVER} in {t_miqp:.2f} s  |  {status}  |  objective {obj_w(w_miqp):.6f}")

# exhaustive verification
def qp_subset(idx):
    Ss, ms = Sigma[np.ix_(idx, idx)], mu[idx]
    f = lambda x: 0.5 * LAMBDA * x @ Ss @ x - ms @ x
    g = lambda x: LAMBDA * Ss @ x - ms
    r = minimize(f, np.full(len(idx), 1 / len(idx)), jac=g, method="SLSQP", bounds=[(max(W_MIN, 0), 1)] * len(idx),
                 constraints=[{"type": "eq", "fun": lambda x: x.sum() - 1, "jac": lambda x: np.ones(len(x))}],
                 options={"ftol": 1e-14, "maxiter": 500})
    return r.fun, r.x

best_enum = None
if ENUM_CHECK:
    t0 = time.time()
    res = [(qp_subset(np.array(c)), c) for c in combinations(range(N), K)]
    (f_en, x_en), c_en = min(res, key=lambda t: t[0][0])
    w_en = np.zeros(N); w_en[list(c_en)] = x_en
    best_enum = (f_en, c_en)
    print(f"Exhaustive check: {len(res)} subsets x convex QP in {time.time() - t0:.1f} s  |  best objective {f_en:.6f}  |  "
          f"same stocks as MIQP: {set(c_en) == set(z_miqp)}  |  objective difference {abs(f_en - obj_w(w_miqp)):.2e}")

# comparison with the equal-weight QUBO solution
if comb(N, K) <= 300000:
    allc = np.array(list(combinations(range(N), K)))
    fe = (0.5 * LAMBDA * Sigma[allc[:, :, None], allc[:, None, :]].sum(axis=(1, 2)) / K ** 2 - mu[allc].sum(axis=1) / K)
    c_eq = allc[np.argmin(fe)]
    w_eq = np.zeros(N); w_eq[c_eq] = 1 / K              # exact optimum of the equal-weight (QUBO) formulation
else:
    w_eq = None

# result
held = np.flatnonzero(w_miqp > 1e-6)
print("\n" + "=" * 78)
print(f"MIQP RESULT (k = {K}): " + ",  ".join(f"{tickers[i]} {w_miqp[i] * 100:.2f}%" for i in held[np.argsort(-w_miqp[held])]))
print("=" * 78)
if len(held) < K:
    print(f"Note: z selects {K} stocks but only {len(held)} receive a positive weight (the model allows w_i = 0 for a selected stock).")
tab = pd.DataFrame({"Annualised return (%)": mu * 100, "Volatility (%)": np.sqrt(np.diag(Sigma)) * 100,
                    "MIQP weight (%)": w_miqp * 100}, index=tickers)
tab["Selected (z=1)"] = np.where(np.isin(np.arange(N), z_miqp), "YES", "-")
display(tab[tab["Selected (z=1)"] == "YES"].sort_values("MIQP weight (%)", ascending=False).round(3))

rows = {f"MIQP (k = {K}, optimal weights)": w_miqp}
if w_eq is not None:
    rows[f"QUBO-type equal weight (k = {K}, exact optimum)"] = w_eq
rows.update({"Equal weight, all 20 stocks": w_ew, "Markowitz max-Sharpe": w_maxsh, "Markowitz min-variance": w_minvar})
cmp_rows = {}
for name, w_ in rows.items():
    r_, v_, s_, h_ = port_stats(w_)
    cmp_rows[name] = {"Exp. return (%)": r_ * 100, "Variance": v_, "Volatility (%)": s_ * 100, "Sharpe": h_,
                      "Objective": obj_w(w_), "No. of stocks": int((w_ > 1e-6).sum())}
CMP = pd.DataFrame(cmp_rows).T
print("\nFull-period risk / return (same mu, Sigma and lambda for every row):")
display(CMP.round(4))
CMP.to_csv(f"{OUT_DIR}/miqp_k{K}_comparison.csv"); tab.to_csv(f"{OUT_DIR}/miqp_k{K}_weights.csv")

# charts
sig2 = np.diag(Sigma)
fig, axes = plt.subplots(1, 2, figsize=(18, 8), gridspec_kw={"width_ratios": [1.25, 1]})
ax = axes[0]
ax.plot(xs(f_var), f_ret, color="red", lw=2.5, label=f"Efficient frontier (long-only, cap {MAX_W:.0%})", zorder=2)
ax.scatter(xs(sig2[w_miqp == 0]), mu[w_miqp == 0], s=60, color="lightgrey", edgecolor="grey", label="Stocks not held", zorder=3)
ax.scatter(xs(sig2[held]), mu[held], s=110, color="#2ca02c", edgecolor="black", label="Stocks held by MIQP", zorder=4)
for i, t in enumerate(tickers):
    ax.annotate(t, (xs(sig2[i]), mu[i]), xytext=(5, 4), textcoords="offset points", fontsize=8)
if w_eq is not None:
    r_, v_, _, _ = port_stats(w_eq)
    ax.scatter(xs(v_), r_, marker="D", s=170, color="#1f77b4", edgecolor="black", zorder=5, label=f"Equal-weight {K} stocks (QUBO-type, exact)")
r_, v_, _, _ = port_stats(w_miqp)
ax.scatter(xs(v_), r_, marker="*", s=650, color="gold", edgecolor="black", linewidths=1.5, zorder=6, label=f"MIQP portfolio (k = {K}, free weights)")
ax.set_xlabel(XLABEL); ax.set_ylabel("Annualised expected return"); ax.grid(True, alpha=0.3); ax.legend(loc="lower right")
ax.set_title("Risk-return map: the MIQP portfolio", fontweight="bold")

ax = axes[1]
idx = np.array(sorted(set(held) | (set(np.flatnonzero(w_eq)) if w_eq is not None else set())))
x_ = np.arange(len(idx)); bw = 0.38
ax.bar(x_ - bw / 2, w_miqp[idx] * 100, bw, color="gold", edgecolor="black", label="MIQP weight")
if w_eq is not None:
    ax.bar(x_ + bw / 2, w_eq[idx] * 100, bw, color="#1f77b4", edgecolor="black", label="Equal weight (QUBO-type)")
ax.set_xticks(x_); ax.set_xticklabels([tickers[i] for i in idx], rotation=45, ha="right")
ax.set_ylabel("Portfolio weight (%)"); ax.grid(True, axis="y", alpha=0.3); ax.legend()
ax.set_title("Weights of the chosen stocks", fontweight="bold")
fig.suptitle(f"MIQP: best {K} of {N} US stocks, {returns.index[0].year}-{returns.index[-1].year}  (lambda = {LAMBDA:.3f}, solver {SOLVER})",
             fontsize=15, fontweight="bold")
plt.tight_layout(); savefig(f"miqp_k{K}_selection"); plt.show()

## Section 8. Ising spin-glass model

The QUBO of Section 5 is rewritten as an **Ising Hamiltonian** and solved. Everything that is used is the same as in the QUBO sections: same data, $\mu$, $\Sigma$, $\lambda$, $k=k^\star$ and $P=P_{k^\star}$. Any difference in the results below therefore comes only from the Ising formulation or from the solver.

### 8.1 QUBO to Ising (the mathematics)

Substituting $x_i=(1+s_i)/2$, with spins $s_i\in\{-1,+1\}$, into $H(x)=x^\top Qx+\text{const}$ (symmetric $Q$, linear terms on the diagonal) gives

$$H(s)=\sum_{i<j}J_{ij}\,s_is_j+\sum_i h_i\,s_i+C,\qquad J_{ij}=\tfrac12 Q_{ij},\quad h_i=\tfrac12\sum_j Q_{ij},\quad C=\tfrac14\sum_{ij}Q_{ij}+\tfrac14\sum_i Q_{ii}+Pk^2 .$$

Bit 1 (stock selected) $\leftrightarrow$ spin $+1$ (up). The constraint "exactly $k$ stocks" becomes a **fixed magnetisation** $\sum_i s_i=2k-N$.

With $Q$ of Section 5.1 the couplings and fields split into three parts (this is checked numerically in the next cell):

$$J_{ij}=\frac P2+\frac{\lambda}{4k^2}\Sigma_{ij},\qquad h_i=\frac{P(N-2k)}{2}-\frac{\mu_i}{2k}+\frac{\lambda}{4k^2}\sum_j\Sigma_{ij}.$$

The pair coupling $P/2$ and the field part $P(N-2k)/2$ come from the penalty and are the same for every pair / every spin; the return $\mu_i$ and the covariance $\Sigma_{ij}$ make the couplings and fields differ between stocks.

**What the next cell does**

* imports `dimod` and the `dwave-samplers` simulated-annealing sampler;
* rebuilds the QUBO matrix $Q$ and constant for $k=k^\star$ with the function `build_Q(P)` (and checks it equals the QUBO used in Section 6.4);
* converts it with the hand-coded formulas above (`qubo_to_ising_manual`);
* prints the number of positive / negative couplings and fields, their ranges, the split of $J$ and $h$ into penalty / return / covariance parts, the number of **frustrated triangles** (triples with $J_{ij}J_{jl}J_{il}>0$, which cannot all be satisfied at once) and the fixed magnetisation;
* shows the table of $h_i$ with $\mu_i$, and the coupling matrix $J$.

In [ ]:
# SECTION 8.1 | QUBO (k = K*) -> Ising: hand-coded conversion
dimod = _ensure("dimod", "dimod")
_ensure("dwave.samplers", "dwave-samplers")
from dwave.samplers import SimulatedAnnealingSampler

M_FIXED = 2 * K - N                         # fixed magnetisation: sum(s) = 2k - N
P = float(P_K[K - 1])                       # penalty of Section 5.4 for this k

def build_Q(Pm):
    """QUBO matrix and constant of Section 5.1 for k = K*:  H(x) = x'Qx + CONST."""
    Qm = 0.5 * LAMBDA / K ** 2 * Sigma - np.diag(mu / K) + Pm * (np.ones((N, N)) - 2 * K * np.eye(N))
    return Qm, Pm * K ** 2

Q_prev = Q.copy()                           # the QUBO sampled in Section 6.4
Q, CONST = build_Q(P)
print(f"lambda = {LAMBDA:.4f} | P (k = {K}) = {P:.4f} | k = {K} | N = {N}")
print(f"QUBO identical to the one of Section 6.4: {np.allclose(Q, Q_prev)}")

def qubo_to_ising_manual(Q, const):
    J = np.triu(Q / 2.0, 1)                    # J_ij = Q_ij / 2   (i < j)
    h = 0.5 * Q.sum(axis=1)                    # h_i  = 1/2 * sum_j Q_ij
    C = 0.25 * Q.sum() + 0.25 * np.trace(Q) + const
    return J, h, C

J, h, C = qubo_to_ising_manual(Q, CONST)
iu = np.triu_indices(N, 1)
Jp = J[iu]
print(f"\n{len(iu[0])} spin-spin couplings (fully connected, N = {N})")
print(f"J_ij : {(Jp > 0).sum()} positive (antiferromagnetic), {(Jp < 0).sum()} negative (ferromagnetic) | "
      f"range [{Jp.min():.5f}, {Jp.max():.5f}] | mean {Jp.mean():.5f} | std {Jp.std():.5f}")
J_pen, J_cov = P / 2, LAMBDA / (4 * K ** 2) * Sigma
assert np.allclose(Jp, J_pen + J_cov[iu])
print(f"  J_ij = P/2 + lambda/(4k^2) Sigma_ij (verified): penalty part P/2 = {J_pen:.5f} for every pair "
      f"({J_pen / Jp.mean() * 100:.0f}% of the mean coupling); covariance part in [{J_cov[iu].min():.5f}, {J_cov[iu].max():.5f}]")
print(f"h_i  : {(h > 0).sum()} positive, {(h < 0).sum()} negative | range [{h.min():.5f}, {h.max():.5f}]")
h_pen, h_ret, h_cov = P * (N - 2 * K) / 2, -mu / (2 * K), LAMBDA / (4 * K ** 2) * Sigma.sum(axis=1)
assert np.allclose(h, h_pen + h_ret + h_cov)
print(f"  h_i = P(N-2k)/2 - mu_i/(2k) + lambda/(4k^2) sum_j Sigma_ij (verified): penalty part {h_pen:.5f} for every spin | "
      f"return part in [{h_ret.min():.5f}, {h_ret.max():.5f}] | covariance part in [{h_cov.min():.5f}, {h_cov.max():.5f}]")
Js = J + J.T
n_tri = len(list(itertools.combinations(range(N), 3)))
frustrated = sum(Js[i, j] * Js[j, l] * Js[i, l] > 0 for i, j, l in itertools.combinations(range(N), 3))
print(f"frustrated triangles (J_ij J_jl J_il > 0): {frustrated} of {n_tri}")
print(f"constant C = {C:.6f}")
print(f"fixed magnetisation: sum(s) = 2k - N = {M_FIXED}  ({K} spins up, {N - K} down)")

print("\nLocal fields h_i, expected returns mu_i and the row sums of J:")
display(pd.DataFrame({"h_i": h, "mu_i (%)": mu * 100, "sum_j J_ij": Js.sum(axis=1)}, index=tickers).round(5))
print("Couplings J (upper triangle, from covariance + penalty):")
display(pd.DataFrame(J, index=tickers, columns=tickers).round(4))

### 8.2 Build the Ising model with dimod and validate it

Two independent routes must give the same Ising model:

* **Route A**: the `dimod` `BinaryQuadraticModel` made from the QUBO and converted with `bqm.to_ising()` (this model is the one sampled later);
* **Route B**: the hand-coded formulas of Section 8.1.

**What the next cell does**

* compares $h$, $J$ and $C$ of both routes (they must agree to $10^{-12}$);
* defines `qubo_energy` and `ising_energy` and runs two validations:
  1. over **all $2^{20}=1\,048\,576$ states**: $E_{\text{Ising}}(s)=E_{\text{QUBO}}(x)$ with $x=(1+s)/2$;
  2. over the **$\binom{20}{k}$ feasible states** (exactly $k$ stocks): $E_{\text{Ising}}$ equals the Markowitz objective $\tfrac{\lambda}{2}w^\top\Sigma w-\mu^\top w$ of the equal-weight portfolio $w=x/k$.

An `assert` stops the notebook if any check fails.

In [ ]:
# SECTION 8.2 | Route A: dimod BQM -> to_ising(), comparison with the hand-coded route B, and validation
bqm_qubo = dimod.BinaryQuadraticModel.from_qubo(
    {(i, j): (Q[i, j] + Q[j, i] if i != j else Q[i, i]) for i in range(N) for j in range(i, N)}, offset=CONST)
h_d, J_d, off_d = bqm_qubo.to_ising()
bqm_ising = dimod.BinaryQuadraticModel.from_ising(h_d, J_d, off_d)          # the SPIN model that is sampled later

# compare with the hand-coded Route B (dimod may store a pair as (i,j) or (j,i))
Jd_mat = np.zeros((N, N))
for (i, j), v in J_d.items():
    Jd_mat[min(i, j), max(i, j)] = v
d_h = max(abs(h_d[i] - h[i]) for i in range(N)); d_J = np.abs(Jd_mat - J).max(); d_C = abs(off_d - C)
print(f"dimod vs hand-coded:  max|dh| = {d_h:.1e} | max|dJ| = {d_J:.1e} | |dC| = {d_C:.1e}")
assert d_h < 1e-12 and d_J < 1e-12 and d_C < 1e-12
print("Route A and Route B agree: h, J and C are verified.\n")

def qubo_energy(X):  return np.einsum("ui,ij,uj->u", X, Q, X) + CONST
def ising_energy(S, J=J, h=h, C=C):
    """S: (m, N) array of +-1 spins -> energies (m,).   H = sum_{i<j} J_ij s_i s_j + sum_i h_i s_i + C"""
    return S @ h + ((S @ J) * S).sum(axis=1) + C

# Validation 1: all 2^20 states, Ising energy = QUBO energy (chunks of 2^16)
t0 = time.time(); max_diff = 0.0; E_min, s_min = np.inf, None
for start in range(0, 2 ** N, 2 ** 16):
    idx = np.arange(start, start + 2 ** 16)
    X = ((idx[:, None] >> np.arange(N)) & 1).astype(float); S = 2 * X - 1
    Eq, Ei = qubo_energy(X), ising_energy(S)
    max_diff = max(max_diff, np.abs(Eq - Ei).max())
    if Ei.min() < E_min: E_min, s_min = Ei.min(), X[Ei.argmin()].copy()
print(f"Validation 1 - all {2 ** N:,} states: max |E_Ising - E_QUBO| = {max_diff:.2e}   ({time.time() - t0:.1f} s)")
assert max_diff < 1e-10

# Validation 2: the 38,760 feasible states (exactly 6 stocks): E_Ising = Markowitz objective 1/2*lam*w'Sigma w - mu'w
combos = np.array(list(itertools.combinations(range(N), K)))
Xf = np.zeros((len(combos), N)); Xf[np.arange(len(combos))[:, None], combos] = 1.0
Wf = Xf / K
obj_f = 0.5 * LAMBDA * np.einsum("ui,ij,uj->u", Wf, Sigma, Wf) - Wf @ mu
Ef = ising_energy(2 * Xf - 1)
print(f"Validation 2 - {len(combos):,} feasible states: max |E_Ising - Markowitz objective| = {np.abs(Ef - obj_f).max():.2e}")
assert np.abs(Ef - obj_f).max() < 1e-10
print("Verified Ising Hamiltonian.")

### 8.3 Exact ground state (brute force)

The constraint fixes the magnetisation, so the exact reference is the minimum of $H(s)$ over all states with $\sum_i s_i=2k-N$, i.e. over all $\binom{N}{k}$ ways to put $k$ spins up. The global minimum over all $2^{20}$ states (from the validation above) is used as a cross-check.

**What the next cell does**

* enumerates every state with $k$ spins up, computes its Ising energy and finds the **ground state** (the optimal portfolio) `S_GS`, its energy `E_GS` and the Markowitz objective `F_EX`, which are equal;
* prints the gap between the ground state and the first excited state, checks that the state is also the global minimum over all $2^{20}$ states and the same portfolio as the QUBO exhaustive optimum;
* shows the 10 lowest-energy states with their gaps to the ground state.

In [ ]:
# SECTION 8.3 | exact ground state
t0 = time.time()
combos_ex = np.array(list(itertools.combinations(range(N), K)))             # every way to put 6 spins up
S_ex = -np.ones((len(combos_ex), N)); S_ex[np.arange(len(combos_ex))[:, None], combos_ex] = 1.0   # all states with sum(s) = -8
E_feas = ising_energy(S_ex)                                                  # Ising energy of each
i_gs = int(np.argmin(E_feas))
x_gs = Xf[i_gs]; sel_gs = np.flatnonzero(x_gs)
t_exact = time.time() - t0
assert np.allclose(E_feas, Ef)
E_GS = float(E_feas[i_gs]); F_EX = float(obj_f[i_gs])
E_sorted = np.sort(E_feas)

# the QUBO exhaustive optimum (same 38,760 states) for the comparison
i_q = int(np.argmin(qubo_energy(Xf)))
print(f"Ising ground state (sum s = {M_FIXED}): " + ", ".join(tickers[i] for i in sel_gs))
print(f"ground-state energy      : {E_GS:.6f}")
print(f"Markowitz objective      : {F_EX:.6f}   (equal to the Ising energy)")
print(f"gap to first excited state: {E_sorted[1] - E_sorted[0]:.6f}")
print(f"same state as global minimum over all 2^20 spin states : {np.array_equal(s_min, x_gs)} (E = {E_min:.6f})")
print(f"same portfolio as the QUBO exhaustive optimum          : {i_q == i_gs}")
print(f"enumeration time: {t_exact * 1000:.1f} ms")
assert np.array_equal(s_min, x_gs) and i_q == i_gs

S_GS = S_ex[i_gs]                                                            # ground-state spin configuration
order_ex = np.argsort(E_feas)
top10 = pd.DataFrame({"Spins up (stocks)": [", ".join(tickers[i] for i in combos_ex[o]) for o in order_ex[:10]],
                      "Ising energy": E_feas[order_ex[:10]],
                      "Gap to ground state": E_feas[order_ex[:10]] - E_GS})
print("\nThe 10 lowest-energy states with sum(s) = " + str(M_FIXED) + ":")
display(top10.round(6))

### 8.4 Ising simulated annealing

**8.4a. D-Wave simulated-annealing sampler on the spin model.** Metropolis single-spin-flip annealing on $h$ and $J$, using the `dwave-samplers` `SimulatedAnnealingSampler` on the **SPIN** model (`bqm_ising`), with the same budget as the QUBO run of Section 6.4 (1000 reads $\times$ 1000 sweeps, seed 42). The QUBO run is repeated alongside it so the two are compared like for like.

**What the next cell does**: samples the Ising model and the QUBO with the same budget and seed, checks that the energy reported by the sampler equals `ising_energy` of the returned spins, and prints the lowest energy found next to the exact one.

In [ ]:
# SECTION 8.4a | Ising simulated annealing with the D-Wave sampler
NUM_READS, NUM_SWEEPS = 1000, 1000
col_order = lambda ss: [list(ss.variables).index(i) for i in range(N)]

t0 = time.time()
ss_ising = SimulatedAnnealingSampler().sample(bqm_ising, num_reads=NUM_READS, num_sweeps=NUM_SWEEPS, seed=SEED)
t_ising = time.time() - t0
t0 = time.time()
ss_qubo = SimulatedAnnealingSampler().sample(bqm_qubo, num_reads=NUM_READS, num_sweeps=NUM_SWEEPS, seed=SEED)   # QUBO benchmark run (same as Section 6.4)
t_qubo = time.time() - t0

S_ising = np.array(ss_ising.record.sample)[:, col_order(ss_ising)].astype(float)      # spins in {-1,+1}
E_ising_reads = np.array(ss_ising.record.energy)
assert np.allclose(E_ising_reads, ising_energy(S_ising), atol=1e-9), "sampler energy != Ising energy"
print(f"Ising SA  : {NUM_READS} reads x {NUM_SWEEPS} sweeps in {t_ising:.2f} s | lowest energy found {E_ising_reads.min():.6f} (exact {E_GS:.6f})")
print(f"QUBO SA   : {NUM_READS} reads x {NUM_SWEEPS} sweeps in {t_qubo:.2f} s (D-Wave sampler on the QUBO, re-run)")
print(f"QUBO SA run of Section 6.4 (same budget and seed): {DWAVE_QUBO_HIT:.1f}% of reads at the exact optimum")

### 8.5 Decode and evaluate the Ising sampler run

The spins are converted back with $x=(1+s)/2$; only samples with exactly $k$ stocks are kept. Every portfolio is scored with `port_stats` and the objective `obj_w`, exactly as for the QUBO.

**What the next cell does**: computes, for the Ising run and for the QUBO run, the share of reads with exactly $k$ stocks and the share that reach the exact optimum, then prints the best Ising-SA portfolio (stocks, return, variance, volatility, Sharpe, objective) and whether it equals the exact ground state.

In [ ]:
# SECTION 8.5 | decode + evaluate
def score_reads(X):
    """X: reads x N bits. Returns feasible mask and the equal-weight Markowitz objective of every read."""
    feas = X.sum(axis=1) == K
    W = X / K
    obj = 0.5 * LAMBDA * np.einsum("ui,ij,uj->u", W, Sigma, W) - W @ mu
    return feas, obj

X_ising = (1 + S_ising) / 2                                   # x = (1 + s)/2
feas_i, obj_i = score_reads(X_ising)
hit_i = feas_i & (obj_i <= F_EX + 1e-10)

best_i = int(np.argmin(np.where(feas_i, obj_i, np.inf)))
w_ising = X_ising[best_i] / K
r_i, v_i, s_i, sh_i = port_stats(w_ising)
obj_best_i = obj_w(w_ising)

# QUBO benchmark run decoded the same way
cq = col_order(ss_qubo); X_q = np.array(ss_qubo.record.sample)[:, cq].astype(float)
feas_q, obj_q = score_reads(X_q); hit_q = feas_q & (obj_q <= F_EX + 1e-10)

print(f"Ising SA (seed {SEED}): reads with exactly {K} stocks {feas_i.mean() * 100:.1f}% | reads at the exact optimum {hit_i.mean() * 100:.1f}% | distinct states {len(np.unique(X_ising, axis=0))}")
print(f"QUBO SA  (seed {SEED}): reads with exactly {K} stocks {feas_q.mean() * 100:.1f}% | reads at the exact optimum {hit_q.mean() * 100:.1f}%   <- QUBO benchmark of Section 6.4")
print("=" * 78)
print("Ising SA best portfolio: " + ", ".join(tickers[i] for i in np.flatnonzero(X_ising[best_i])) + f"  -> {100 / K:.2f}% each")
print(f"return {r_i * 100:.2f}% | variance {v_i:.4f} | volatility {s_i * 100:.2f}% | Sharpe {sh_i:.3f} | objective {obj_best_i:.4f}")
print(f"same stocks as the exact ground state: {set(np.flatnonzero(X_ising[best_i])) == set(sel_gs)}")
print("=" * 78)

r_e, v_e, s_e, sh_e = port_stats(x_gs / K)

### 8.6 Own Ising simulated annealer (single-spin flips and two-spin moves)

Besides the library sampler, the Ising model is annealed with the **own simulated annealer written for spins**:

* local field $f_i=h_i+\sum_j J_{ij}s_j$; flipping $s_i\to-s_i$ costs $\Delta E=-2s_if_i$ (the same energy change as the corresponding bit flip of the QUBO);
* `n_chains` independent chains are annealed in parallel, the temperature falls geometrically over `n_sweeps` sweeps (start $=0.1\times$ the median $|\Delta E|$ of random states, end $=10^{-4}\times$ start, the rule of Section 6.1);
* with `pair_moves=True`, after the single flips of every sweep, $n$ **two-spin moves** are proposed (spins $i$ and $k$ are flipped together). With one up and one down spin this swaps a stock in for a stock out and keeps $\sum_i s_i$ fixed. Their cost is
$$\Delta E=-2s_if_i-2s_kf_k+4J_{ik}s_is_k .$$
These are the same moves as the two-bit moves of the QUBO annealer of Section 6.1.

**What the next cell does**: defines `ising_sa`, runs it once with single flips (128 chains $\times$ 150 sweeps and 1000 chains $\times$ 1000 sweeps) and once with single flips plus two-spin moves (128 $\times$ 150), prints for each run the lowest energy, the share of feasible reads and the share of reads in the exact ground state, and decodes the best portfolio of each run.

In [ ]:
# SECTION 8.6 | Own Ising simulated annealer: Metropolis single-spin flips (+ optional two-spin moves) on h and J
#   local field  f_i = h_i + sum_j J_ij s_j     flip s_i -> -s_i costs   dE = -2 s_i f_i
#   two-spin move cost:  dE = -2 s_i f_i - 2 s_k f_k + 4 J_ik s_i s_k
def ising_sa(J, h, C, n_chains=128, n_sweeps=150, rng=None, t_scale=0.1, pair_moves=False):
    """Returns final spin states (chains x n) and their energies.  J is the upper-triangular coupling matrix."""
    rng = rng if rng is not None else np.random.default_rng(SEED)
    Jf = J + J.T                                                    # full symmetric couplings (zero diagonal)
    n = len(h)
    S = rng.choice([-1.0, 1.0], size=(n_chains, n))
    F = S @ Jf + h                                                  # local fields of every chain
    t_hi = t_scale * np.median(np.abs(-2 * S * F))
    temps = np.geomspace(t_hi, t_hi * 1e-4, n_sweeps)
    for T in temps:
        U = rng.random((n, n_chains))
        for j, i in enumerate(rng.permutation(n)):
            dE = -2 * S[:, i] * F[:, i]
            acc = (dE <= 0) | (U[j] < np.exp(-np.maximum(dE, 0) / T))
            if acc.any():
                m = -2 * S[:, i] * acc                              # change of s_i where the flip is accepted
                S[:, i] += m
                F += m[:, None] * Jf[i]
        if pair_moves:
            U2 = rng.random((n, n_chains))
            pairs = np.array([rng.choice(n, 2, replace=False) for _ in range(n)])
            for j, (i, k) in enumerate(pairs):
                dE = -2 * S[:, i] * F[:, i] - 2 * S[:, k] * F[:, k] + 4 * Jf[i, k] * S[:, i] * S[:, k]
                acc = (dE <= 0) | (U2[j] < np.exp(-np.maximum(dE, 0) / T))
                if acc.any():
                    mi, mk = -2 * S[:, i] * acc, -2 * S[:, k] * acc
                    S[:, i] += mi; S[:, k] += mk
                    F += mi[:, None] * Jf[i] + mk[:, None] * Jf[k]
    return S, ising_energy(S, J, h, C)

# one run of each budget:  single-spin flips, and single flips + two-spin moves
RUNS, RUNS_PAIR = {}, {}
for chains, sweeps, pm in [(128, 150, False), (1000, 1000, False), (128, 150, True)]:
    t0 = time.time()
    S_run, E_run = ising_sa(J, h, C, chains, sweeps, rng=np.random.default_rng(SEED), pair_moves=pm)
    dt = time.time() - t0
    (RUNS_PAIR if pm else RUNS)[(chains, sweeps)] = (S_run, E_run, dt)
    feas = S_run.sum(axis=1) == M_FIXED
    hit = np.isclose(E_run, E_GS, atol=1e-10)
    print(f"{chains:>4} chains x {sweeps:>4} sweeps, {'+ two-spin moves' if pm else 'single flips   '}: {dt:5.2f} s | "
          f"lowest energy {E_run.min():.6f} (exact {E_GS:.6f}) | feasible reads {feas.mean() * 100:5.1f}% | "
          f"reads at the ground state {hit.mean() * 100:5.1f}%")

# decode (x = (1 + s) / 2) with the SAME decode_select / obj_w as the QUBO
def decode_spins(S):
    return (1 + S) / 2
q_k = dict(k=K)
w_ising_sa, info_sa = {}, {}
for key, (S_run, E_run, dt) in RUNS.items():
    w_ising_sa[key], info_sa[key] = decode_select(decode_spins(S_run), E_run, q_k, mu, Sigma, LAMBDA)
w_ising_pair, _ = decode_select(decode_spins(RUNS_PAIR[(128, 150)][0]), RUNS_PAIR[(128, 150)][1], q_k, mu, Sigma, LAMBDA)
print("\nbest portfolio of each run:")
for key in RUNS:
    print(f"  single flips {key[0]:>4} x {key[1]:>4}: " + ", ".join(tickers[i] for i in np.flatnonzero(w_ising_sa[key])) + f"  | objective {obj_w(w_ising_sa[key]):.6f}")
print("  single flips + two-spin moves 128 x  150: " + ", ".join(tickers[i] for i in np.flatnonzero(w_ising_pair)) + f"  | objective {obj_w(w_ising_pair):.6f}")
print(f"exact ground state objective {F_EX:.6f}")

### 8.7 Comparison experiments: D-Wave sampler, Ising versus QUBO, 30 seeds

Each stochastic solver is repeated over **30 random seeds** (1000 reads $\times$ 1000 sweeps per seed). Metrics: best objective, gap to the exact optimum, % of reads that hit the optimum, % of feasible reads, runtime. The QUBO model (the D-Wave sampler of Section 6.4) is run with the same seeds as a like-for-like reference.

**What the next cell does**

* **8.7a** headline comparison: Ising SA versus QUBO SA, 1000 reads $\times$ 1000 sweeps, 30 seeds;
* **8.7b** sensitivity to the **number of sweeps** (10, 50, 150, 1000) for the Ising model;
* **8.7c** sensitivity to the **penalty** $P$ (0.5$\times$, 1$\times$, 2$\times$ of the Monte-Carlo value) for the Ising model with 1000 sweeps.

The next cells are slower than the previous ones because many annealing runs are repeated.

In [ ]:
# SECTION 8.7 | experiments with the D-Wave sampler (30 seeds)
SEEDS = list(range(1, 31))

def run_config(bqm, seeds, sweeps, reads=1000):
    rows = []
    for sd in seeds:
        t0 = time.time()
        ss = SimulatedAnnealingSampler().sample(bqm, num_reads=reads, num_sweeps=sweeps, seed=sd)
        dt = time.time() - t0
        Z = np.array(ss.record.sample)[:, col_order(ss)].astype(float)
        X = (1 + Z) / 2 if ss.vartype is dimod.SPIN else Z
        feas, obj = score_reads(X)
        best = obj[feas].min() if feas.any() else np.nan
        rows.append({"seed": sd, "Best objective": best, "Gap to exact": best - F_EX,
                     "Reads at optimum (%)": (feas & (obj <= F_EX + 1e-10)).mean() * 100,
                     "Feasible reads (%)": feas.mean() * 100, "Runtime (s)": dt,
                     "Best read = optimum": bool(feas.any() and best <= F_EX + 1e-10)})
    return pd.DataFrame(rows)

def summarise(df):
    return pd.Series({"Best objective (mean)": df["Best objective"].mean(),
                      "Gap to exact (mean)": df["Gap to exact"].mean(),
                      "Reads at optimum % (mean)": df["Reads at optimum (%)"].mean(),
                      "Reads at optimum % (std)": df["Reads at optimum (%)"].std(ddof=1),
                      "Reads at optimum % (min-max)": f"{df['Reads at optimum (%)'].min():.1f} - {df['Reads at optimum (%)'].max():.1f}",
                      "Feasible reads % (mean)": df["Feasible reads (%)"].mean(),
                      "Feasible reads % (std)": df["Feasible reads (%)"].std(ddof=1),
                      f"Seeds whose best read is optimal (of {len(df)})": int(df["Best read = optimum"].sum()),
                      "Runtime s (mean)": df["Runtime (s)"].mean()})

# 6a. headline: 1000 reads x 1000 sweeps, 30 seeds, Ising vs QUBO (same seeds)
res_ising = run_config(bqm_ising, SEEDS, 1000)
res_qubo  = run_config(bqm_qubo,  SEEDS, 1000)
tab_main = pd.DataFrame({"Ising SA (SPIN model)": summarise(res_ising), "QUBO SA (D-Wave sampler on the QUBO)": summarise(res_qubo)})
print("6a. 30 seeds x (1000 reads x 1000 sweeps), exact objective =", round(F_EX, 6))
display(tab_main)

# 6b. sensitivity to the number of sweeps (Ising model, P = 1x)
SWEEPS = [10, 50, 150, 1000]
res_sweeps = {sw: run_config(bqm_ising, SEEDS, sw) for sw in SWEEPS}
tab_sweeps = pd.DataFrame({sw: summarise(df) for sw, df in res_sweeps.items()}).T.rename_axis("sweeps")
print("6b. Sensitivity to the number of sweeps (Ising, 1000 reads, 30 seeds)")
display(tab_sweeps[["Reads at optimum % (mean)", "Reads at optimum % (std)", "Feasible reads % (mean)",
                    f"Seeds whose best read is optimal (of {len(SEEDS)})", "Gap to exact (mean)", "Runtime s (mean)"]])

# 6c. sensitivity to the penalty P (Ising model, 1000 sweeps)
def ising_bqm_for_P(Pm):
    Qm, Cm = build_Q(Pm)
    b = dimod.BinaryQuadraticModel.from_qubo({(i, j): (Qm[i, j] + Qm[j, i] if i != j else Qm[i, i])
                                              for i in range(N) for j in range(i, N)}, offset=Cm)
    hh, JJ, oo = b.to_ising()
    return dimod.BinaryQuadraticModel.from_ising(hh, JJ, oo)

P_MULTS = [0.5, 1.0, 2.0]
res_P = {m: run_config(ising_bqm_for_P(P * m), SEEDS, 1000) for m in P_MULTS}
tab_P = pd.DataFrame({f"{m}x (P = {P * m:.4f})": summarise(df) for m, df in res_P.items()}).T.rename_axis("penalty")
print("6c. Sensitivity to the penalty P (Ising, 1000 reads x 1000 sweeps, 30 seeds)")
display(tab_P[["Reads at optimum % (mean)", "Reads at optimum % (std)", "Feasible reads % (mean)",
               f"Seeds whose best read is optimal (of {len(SEEDS)})", "Gap to exact (mean)", "Runtime s (mean)"]])

### 8.8 Comparison experiments: own annealer, 30 seeds, and sensitivity

The same kind of experiment for the **own** annealers of Section 6.1 (QUBO) and Section 8.6 (Ising). Every solver is repeated over 30 seeds (`SEED`, `SEED+1`, …) and the same metrics are collected.

**What the next two cells do**

* **8.8a** runs the own Ising annealer with single flips (128 $\times$ 150 and 1000 $\times$ 1000), the own Ising annealer with two-spin moves (128 $\times$ 150) and the own QUBO annealer of Section 6.1 (128 $\times$ 150). A read counts as "at the optimum" when it equals the exact optimal configuration `S_GS`.
* **8.8b** sensitivity of the own Ising annealer (single flips, 1000 reads, 30 seeds) to the number of sweeps and to the penalty $P$ (0.5$\times$, 1$\times$, 2$\times$). For each $P$ the **exact ground state of that Hamiltonian is found by brute force over all $2^{20}$ spin states**, to check that it still has exactly $k$ stocks and is still the same portfolio.

In [ ]:
# SECTION 8.8a | Comparison experiments with the own annealers: every stochastic solver repeated over 30 seeds
N_SEEDS = 30
f_opt = E_GS                                         # exact optimum (Ising ground state = Markowitz objective of the best k stocks)
is_hit = lambda S: (S == S_GS).all(axis=1)           # read equals the exact optimal configuration

def run_metrics(S, E, dt):
    """S: spins (reads x N). Metrics of one run (one seed)."""
    feas = S.sum(axis=1) == M_FIXED
    w, _ = decode_select(decode_spins(S), E, q_k, mu, Sigma, LAMBDA)
    return {"best objective": obj_w(w), "gap to optimum": obj_w(w) - f_opt, "% reads at optimum": is_hit(S).mean() * 100,
            "% feasible reads": feas.mean() * 100, "best read = optimum": float(obj_w(w) <= f_opt + 1e-10), "runtime (s)": dt}

def repeat(fn):
    rows = []
    for s in range(N_SEEDS):
        t0 = time.time(); S, E = fn(np.random.default_rng(SEED + s)); dt = time.time() - t0
        rows.append(run_metrics(S, E, dt))
    return pd.DataFrame(rows)

RES = {}
RES["Ising SA (own code), 128 chains x 150 sweeps"] = repeat(lambda r: ising_sa(J, h, C, 128, 150, rng=r))
RES["Ising SA (own code), 1000 reads x 1000 sweeps"] = repeat(lambda r: ising_sa(J, h, C, 1000, 1000, rng=r))
RES["Ising SA (own code) + two-spin moves, 128 chains x 150 sweeps"] = repeat(lambda r: ising_sa(J, h, C, 128, 150, rng=r, pair_moves=True))

def qubo_sa_as_spins(r):                              # QUBO annealer of Section 6.1 (bits) -> spins, so the same metrics apply
    X_, E_, _ = simulated_annealing(Q, 128, 150, rng=r)
    return 2 * X_ - 1, E_ + CONST
RES["QUBO SA (own code of Section 6.1), 128 x 150"] = repeat(qubo_sa_as_spins)

summ = pd.DataFrame({name: {"best objective (mean)": d["best objective"].mean(),
                            "gap to optimum (max over seeds)": d["gap to optimum"].max(),
                            "seeds whose best read = optimum": f"{int(d['best read = optimum'].sum())} / {N_SEEDS}",
                            "% reads at optimum (mean)": d["% reads at optimum"].mean(),
                            "% reads at optimum (std)": d["% reads at optimum"].std(),
                            "% feasible reads (mean)": d["% feasible reads"].mean(),
                            "runtime s per run (mean)": d["runtime (s)"].mean()} for name, d in RES.items()}).T
print(f"Exact optimum: objective {f_opt:.6f}.  {N_SEEDS} seeds per solver (seeds {SEED}..{SEED + N_SEEDS - 1}).")
print(f"QUBO benchmark of Section 6.4 (D-Wave sampler, 1000 reads x 1000 sweeps, seed {SEED}): {DWAVE_QUBO_HIT:.1f}% of reads at the optimum.\n")
display(summ.round({"best objective (mean)": 6, "gap to optimum (max over seeds)": 8, "% reads at optimum (mean)": 2,
                    "% reads at optimum (std)": 2, "% feasible reads (mean)": 1, "runtime s per run (mean)": 3}))

In [ ]:
# SECTION 8.8b | Sensitivity of the own Ising annealer (single flips, 1000 reads, 30 seeds) to the sweeps and to the penalty P
def bits(idx, n=N):
    return ((idx[:, None] >> np.arange(n)) & 1).astype(float)

def sa_stats(Jx, hx, Cx, sweeps, n_reads=1000):
    out = []
    for s in range(N_SEEDS):
        S, E = ising_sa(Jx, hx, Cx, n_reads, sweeps, rng=np.random.default_rng(SEED + s))
        out.append((is_hit(S).mean() * 100, (S.sum(axis=1) == M_FIXED).mean() * 100))
    out = np.array(out)
    return out[:, 0].mean(), out[:, 0].std(), out[:, 1].mean()

# (i) number of sweeps (P = 1x)
SWEEPS_OWN = [10, 50, 150, 1000]
sw = pd.DataFrame([dict(zip(["% reads at optimum (mean)", "std", "% feasible reads (mean)"], sa_stats(J, h, C, s_))) for s_ in SWEEPS_OWN],
                  index=pd.Index(SWEEPS_OWN, name="sweeps"))
print("Sweeps (1000 reads, P = 1x):")
display(sw.round(2))

# (ii) penalty P at 0.5x, 1x, 2x (1000 reads x 1000 sweeps); the exact ground state of each Hamiltonian is found by brute force over all 2^20 spins
rows = []
for mult in [0.5, 1.0, 2.0]:
    Qm, Cm = build_Q(mult * P); Jm, hm, Cmm = qubo_to_ising_manual(Qm, Cm)
    e_best, s_best = np.inf, None
    for start in range(0, 1 << N, 1 << 16):
        S = 2 * bits(np.arange(start, start + (1 << 16))) - 1
        e = ising_energy(S, Jm, hm, Cmm); j = int(np.argmin(e))
        if e[j] < e_best: e_best, s_best = float(e[j]), S[j].copy()
    m_, sd_, fe_ = sa_stats(Jm, hm, Cmm, 1000)
    rows.append({"P multiplier": mult, "P": mult * P, "ground state has exactly k stocks": bool(s_best.sum() == M_FIXED),
                 f"ground state = same {K} stocks": bool(np.array_equal(s_best, S_GS)),
                 "% reads at optimum (mean)": m_, "std": sd_, "% feasible reads (mean)": fe_})
pen = pd.DataFrame(rows).set_index("P multiplier")
print("\nPenalty P (1000 reads x 1000 sweeps):")
display(pen.round({"P": 5, "% reads at optimum (mean)": 2, "std": 2, "% feasible reads (mean)": 1}))

### 8.9 Charts and spin-glass diagnostics

**What the next cells draw**

1. **Risk-return map**: all equal-weight $k$-stock portfolios (light blue), the efficient frontier, the QUBO optimum, the Ising ground state, the best Ising-SA sample and the MIQP portfolio (Section 7).
2. **Energy histogram and success rate**: (left) the energies of the 1000 sampled spin configurations of the D-Wave sampler with the exact ground state; (right) share of reads at the exact optimum versus the number of sweeps (mean $\pm$ 1 std over 30 seeds), with the QUBO benchmark.
3. **Weights**: equal-weight Ising / QUBO portfolio versus the free-weight MIQP portfolio.
4. **Own annealer**: histogram of the sampled energies (1000 $\times$ 1000 run) and its success rate versus the number of sweeps.
5. **Spin-glass diagnostics**: heat map of the coupling matrix $J$ and the energy gaps of the 10 lowest-energy states.

In [ ]:
# SECTION 8.9 | chart 1: risk-return map (Ising / QUBO / MIQP vs the efficient frontier)
rc = mu[combos].mean(axis=1); vc = Sigma[combos[:, :, None], combos[:, None, :]].sum(axis=(1, 2)) / K ** 2
w_q = Xf[i_q] / K; r_q, v_q, _, _ = port_stats(w_q)
r_m, v_m, _, _ = port_stats(w_miqp)

fig, ax = plt.subplots(figsize=(13, 8))
ax.scatter(xs(vc), rc, s=8, color="#9ecae1", alpha=0.5, linewidths=0, label=f"All {len(combos):,} possible {K}-stock equal-weight portfolios")
ax.plot(xs(f_var), f_ret, color="red", lw=2.5, label=f"Efficient frontier (long-only, cap {MAX_W:.0%})")
ax.scatter(xs(v_q), r_q, marker="^", s=330, color="#9467bd", edgecolor="black", linewidths=1.2, zorder=5, label="QUBO optimum (exhaustive)")
ax.scatter(xs(v_e), r_e, marker="*", s=700, color="gold", edgecolor="black", linewidths=1.5, zorder=6, label="Ising ground state (exhaustive)")
ax.scatter(xs(v_i), r_i, marker="o", s=110, color="#ff7f0e", edgecolor="black", linewidths=1.2, zorder=7, label="Ising SA best sample")
ax.scatter(xs(v_m), r_m, marker="D", s=170, color="#2ca02c", edgecolor="black", linewidths=1.2, zorder=6, label=f"MIQP ({SOLVER}, free weights)")
ax.set_xlabel(XLABEL); ax.set_ylabel("Annualised expected return"); ax.grid(True, alpha=0.3); ax.legend(loc="lower right")
ax.set_title(f"Ising ground state vs QUBO and MIQP on the risk-return plane (k = {K}, lambda = {LAMBDA:.3f})", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("ising_risk_return_map"); plt.show()

In [ ]:
# SECTION 8.9 | chart 2: energy histogram of the sampled Ising energies + success rate versus sweeps
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
ax = axes[0]
bins = np.linspace(E_ising_reads.min(), np.percentile(E_ising_reads, 99), 40)
ax.hist(E_ising_reads[feas_i], bins=bins, color="#2ca02c", alpha=0.8, label=f"exactly {K} stocks (sum s = {M_FIXED})")
if (~feas_i).any():
    ax.hist(E_ising_reads[~feas_i], bins=bins, color="#d62728", alpha=0.6, label="wrong number of stocks")
ax.axvline(E_GS, color="gold", lw=3, label=f"exact ground state ({E_GS:.4f})")
ax.set_xlabel("Ising energy of the sample"); ax.set_ylabel(f"number of reads (of {NUM_READS})"); ax.grid(True, alpha=0.3); ax.legend()
ax.set_title(f"Energies of the sampled spin configurations (seed {SEED}, {NUM_SWEEPS} sweeps)", fontweight="bold")

ax = axes[1]
m = np.array([res_sweeps[s]["Reads at optimum (%)"].mean() for s in SWEEPS]); sd = np.array([res_sweeps[s]["Reads at optimum (%)"].std(ddof=1) for s in SWEEPS])
ax.plot(SWEEPS, m, marker="o", color="#1f77b4", lw=2, label="reads at the exact optimum, mean over 30 seeds")
ax.fill_between(SWEEPS, np.maximum(m - sd, 0), m + sd, color="#1f77b4", alpha=0.2, label="+/- 1 std over seeds")
ax.axhline(DWAVE_QUBO_HIT, color="grey", ls="--", lw=1.2, label=f"QUBO SA benchmark of Section 6.4 ({DWAVE_QUBO_HIT:.1f}%)")
ax.set_xscale("log"); ax.set_xticks(SWEEPS); ax.set_xticklabels(SWEEPS)
ax.set_xlabel("sweeps per read"); ax.set_ylabel("% of reads at the exact optimum"); ax.grid(True, alpha=0.3)
ax.set_ylim(0, max(5, 1.15 * (m + sd).max())); ax.legend(loc="upper left", fontsize=9)
ax.set_title("Success rate versus number of sweeps (Ising SA)", fontweight="bold")
fig.suptitle("Ising simulated annealing diagnostics", fontsize=15, fontweight="bold")
plt.tight_layout(); savefig("ising_energy_histogram_and_sweeps"); plt.show()

In [ ]:
# SECTION 8.9 | chart 3: weights, Ising / QUBO (equal weights) vs MIQP (free weights)
held_m = np.flatnonzero(w_miqp > 1e-6)
stock_idx = sorted(set(held_m) | set(sel_gs))
stocks = [tickers[i] for i in stock_idx]
w_is = [100 / K if i in set(sel_gs) else 0 for i in stock_idx]
w_mq = [w_miqp[i] * 100 for i in stock_idx]
xp = np.arange(len(stocks)); bw = 0.38
fig, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(xp - bw / 2, w_is, bw, color="#ff7f0e", edgecolor="black", label="Ising ground state = QUBO optimum (equal weights)")
b2 = ax.bar(xp + bw / 2, w_mq, bw, color="#2ca02c", edgecolor="black", label="MIQP (free weights)")
for b in list(b1) + list(b2):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.3, f"{b.get_height():.1f}", ha="center", fontsize=9)
ax.set_xticks(xp); ax.set_xticklabels(stocks); ax.set_ylabel("Portfolio weight (%)"); ax.grid(True, axis="y", alpha=0.3); ax.legend()
ax.set_title(f"Portfolio weights: Ising / QUBO vs MIQP (k = {K})", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("ising_weights_vs_miqp"); plt.show()

In [ ]:
# SECTION 8.9 | chart 4: own annealer - sampled Ising energies + success rate versus number of sweeps
S_h, E_h, _ = RUNS[(1000, 1000)]
feas_h = S_h.sum(axis=1) == M_FIXED
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
ax = axes[0]
bins = np.linspace(E_GS - 1e-4, np.percentile(E_h, 99), 40)
ax.hist(E_h[feas_h], bins=bins, color="#2ca02c", alpha=0.8, label=f"sampled Ising energies, exactly {K} stocks (1000 reads x 1000 sweeps)")
if (~feas_h).any():
    ax.hist(E_h[~feas_h], bins=bins, color="#d62728", alpha=0.6, label="wrong number of stocks")
ax.axvline(E_GS, color="gold", lw=1.5, ls="--", label=f"exact ground state  E = {E_GS:.4f}")
ax.set_xlabel("Ising energy of the sample"); ax.set_ylabel("number of reads (of 1000)"); ax.grid(True, alpha=0.3); ax.legend(loc="upper right", fontsize=9)
ax.set_title("Energies of the sampled spin configurations (own annealer, single flips)", fontweight="bold")
ax = axes[1]
ax.errorbar(sw.index, sw["% reads at optimum (mean)"], yerr=sw["std"], marker="o", lw=2, capsize=4, color="#1f77b4", label="Ising SA (own code), single flips (mean +- std, 30 seeds)")
ax.axhline(DWAVE_QUBO_HIT, color="#d62728", ls="--", lw=1.5, label=f"QUBO SA benchmark of Section 6.4: {DWAVE_QUBO_HIT:.1f}%")
ax.set_xscale("log"); ax.set_xticks(sw.index); ax.set_xticklabels(sw.index)
ax.set_xlabel("number of sweeps"); ax.set_ylabel("% of reads at the exact optimum"); ax.grid(True, alpha=0.3); ax.legend()
ax.set_title("Success rate versus number of sweeps", fontweight="bold")
plt.tight_layout(); savefig("ising_own_sa_energy_histogram_and_sweeps"); plt.show()

In [ ]:
# SECTION 8.9 | chart 5: heat map of the coupling matrix J and the energy gaps of the 10 lowest-energy states
fig, axes = plt.subplots(1, 2, figsize=(18, 7.5), gridspec_kw={"width_ratios": [1.1, 1]})
ax = axes[0]
im = ax.imshow(np.ma.masked_where(np.eye(N, dtype=bool), Js), cmap="viridis", vmin=J[iu].min(), vmax=J[iu].max())
ax.set_xticks(range(N)); ax.set_xticklabels(tickers, rotation=90, fontsize=8); ax.set_yticks(range(N)); ax.set_yticklabels(tickers, fontsize=8)
plt.colorbar(im, ax=ax, label="J_ij"); ax.set_title("Coupling matrix J (penalty + covariance)", fontweight="bold")
ax = axes[1]
gaps = top10["Gap to ground state"].to_numpy()
ax.barh(range(10)[::-1], gaps, color="#9467bd", edgecolor="black")
ax.set_yticks(range(10)[::-1]); ax.set_yticklabels([s_.replace(", ", "/") for s_ in top10["Spins up (stocks)"]], fontsize=8)
for yy, g in zip(range(10)[::-1], gaps):
    ax.text(g, yy, f" {g:.4f}", va="center", fontsize=8)
ax.set_xlabel("energy gap to the ground state"); ax.grid(True, axis="x", alpha=0.3)
ax.set_title(f"10 lowest-energy states (sum(s) = {M_FIXED})", fontweight="bold")
plt.tight_layout(); savefig("ising_J_heatmap_and_gaps"); plt.show()

## Section 9. Comparison table and conclusions (objective 5)

### 9.1 Comparison table: simulated annealing and MIQP versus the Ising model (20 US stocks)

Every row is computed in this notebook from the same $\mu$, $\Sigma$, $\lambda$ and $k$, and is scored with the same functions (`port_stats`, `obj_w`). Rows beginning with *Ising* are the Ising results (highlighted).

**What the next cell does**: collects the portfolio of every method (MIQP, QUBO with the own simulated annealer, QUBO with the D-Wave sampler, Ising by exhaustive search, Ising with the D-Wave sampler, Ising with the own annealer with single flips and with two-spin moves), prints its return, volatility, Sharpe ratio and objective, saves the table, and then shows the solver statistics over 30 seeds.

In [ ]:
# SECTION 9.1 | COMPARISON TABLE (20 US stocks): QUBO + simulated annealing, MIQP and Ising
def fmt(w_idx):
    return ", ".join(tickers[i] for i in w_idx)

def row(method, w, extra=""):
    r_, v_, s_, h_ = port_stats(w)
    held_ = [i for i in np.argsort(-w) if w[i] > 1e-6]
    weights_txt = ", ".join(f"{tickers[i]} {w[i] * 100:.1f}%" for i in held_)
    return {"Method": method, "Portfolio": weights_txt + extra, "Return %": r_ * 100, "Vol %": s_ * 100,
            "Sharpe": h_, "Objective": obj_w(w)}

hit_own = {n_: d["% reads at optimum"].mean() for n_, d in RES.items()}
rows_tab = [
    row(f"MIQP ({SOLVER}, k = {K}, free weights)", w_miqp),
    row(f"QUBO + own SA (k = {K})", SOL[K], f"  (optimum in {QK.loc[K, 'Chains at exact optimum (%)']:.1f}% of chains)"),
    row("QUBO + D-Wave local SA", Wmat[best], f"  (optimum in {DWAVE_QUBO_HIT:.1f}% of reads)"),
    row("Ising + exhaustive search", x_gs / K),
    row("Ising + D-Wave SA sampler (SPIN model, 1000 x 1000)", w_ising, f"  (optimum in {hit_i.mean() * 100:.1f}% of reads, seed {SEED})"),
    row("Ising + own SA (single flips, 1000 x 1000)", w_ising_sa[(1000, 1000)], f"  (optimum in {hit_own['Ising SA (own code), 1000 reads x 1000 sweeps']:.1f}% of reads, mean of 30 seeds)"),
    row("Ising + own SA (single flips, 128 x 150)", w_ising_sa[(128, 150)], f"  (optimum in {hit_own['Ising SA (own code), 128 chains x 150 sweeps']:.1f}% of reads, mean of 30 seeds)"),
    row("Ising + own SA with two-spin moves (128 x 150)", w_ising_pair, f"  (optimum in {hit_own['Ising SA (own code) + two-spin moves, 128 chains x 150 sweeps']:.1f}% of reads, mean of 30 seeds)"),
]
cmp_tab = pd.DataFrame(rows_tab).set_index("Method")
cmp_tab.loc["Ising + QPU (quantum annealer)"] = ["not run", np.nan, np.nan, np.nan, np.nan]
cmp_tab.to_csv(f"{OUT_DIR}/comparison_table_20stocks.csv")
def hl(r_):
    return ["background-color: #fff2a8" if r_.name.startswith("Ising") else "" for _ in r_]
display(cmp_tab.style.apply(hl, axis=1).format({"Return %": "{:.2f}", "Vol %": "{:.2f}", "Sharpe": "{:.3f}", "Objective": "{:.4f}"}, na_rep="-")
        .set_properties(subset=["Portfolio"], **{"text-align": "left"}))

print("Solver statistics over 30 seeds - D-Wave sampler, 1000 reads x 1000 sweeps (Section 8.7a):")
display(tab_main)
print("Solver statistics over 30 seeds - own annealers (Section 8.8a):")
display(summ.round({"best objective (mean)": 6, "gap to optimum (max over seeds)": 8, "% reads at optimum (mean)": 2,
                    "% reads at optimum (std)": 2, "% feasible reads (mean)": 1, "runtime s per run (mean)": 3}))
tab_main.to_csv(f"{OUT_DIR}/solver_statistics_dwave_sampler.csv"); summ.to_csv(f"{OUT_DIR}/solver_statistics_own_annealers.csv")
sw.to_csv(f"{OUT_DIR}/ising_sweeps_sensitivity_own_sa.csv"); pen.to_csv(f"{OUT_DIR}/ising_penalty_sensitivity_own_sa.csv")
tab_sweeps.to_csv(f"{OUT_DIR}/ising_sweeps_sensitivity_dwave.csv"); tab_P.to_csv(f"{OUT_DIR}/ising_penalty_sensitivity_dwave.csv")
top10.to_csv(f"{OUT_DIR}/ising_lowest_states.csv")

### 9.2 Conclusions

The next cell writes the conclusions. **Every number in the text is filled in from the results above**, so the statement always matches the run.

In [ ]:
# SECTION 9.2 | conclusions (every number is filled in from the results computed above)
mi, mq = res_ising["Reads at optimum (%)"], res_qubo["Reads at optimum (%)"]
t_i, t_q = res_ising["Runtime (s)"].mean(), res_qubo["Runtime (s)"].mean()
p05, p1, p2 = tab_P.iloc[0], tab_P.iloc[1], tab_P.iloc[2]
sw10, sw1000 = tab_sweeps.loc[10], tab_sweeps.loc[1000]
SEEDCOL = f"Seeds whose best read is optimal (of {len(SEEDS)})"
own_s128 = hit_own["Ising SA (own code), 128 chains x 150 sweeps"]
own_s1000 = hit_own["Ising SA (own code), 1000 reads x 1000 sweeps"]
own_pair = hit_own["Ising SA (own code) + two-spin moves, 128 chains x 150 sweeps"]
own_qubo = hit_own["QUBO SA (own code of Section 6.1), 128 x 150"]
sel_sets = {"QUBO + own SA": set(np.flatnonzero(SOL[K])), "QUBO + D-Wave SA": set(np.flatnonzero(Wmat[best])),
            "Ising exhaustive": set(sel_gs), "Ising D-Wave SA": set(np.flatnonzero(X_ising[best_i])),
            "Ising own SA (single flips)": set(np.flatnonzero(w_ising_sa[(1000, 1000)])),
            "Ising own SA (two-spin moves)": set(np.flatnonzero(w_ising_pair))}
same_equal = len({frozenset(v) for v in sel_sets.values()}) == 1
miqp_set = set(np.flatnonzero(w_miqp > 1e-6))
r_mq_, v_mq_, s_mq_, sh_mq_ = port_stats(w_miqp)
pen_rows = pen.index.tolist()
pen_same = bool(pen[f"ground state = same {K} stocks"].all())
sweeps_txt = ", ".join(f"{s_}: {sw.loc[s_, '% reads at optimum (mean)']:.2f}%" for s_ in SWEEPS_OWN)

display(Markdown(f"""
**Statement on the compared results (20 US stocks, k = {K}, lambda = {LAMBDA:.4f}, P = {P:.4f}).**
{"All equal-weight methods (QUBO with the own and the D-Wave simulated annealer, the Ising ground state, and the Ising simulated annealers) select the same " + str(K) + " stocks: **" + fmt(sel_gs) + "**" if same_equal else "The equal-weight methods do not all select the same stocks (see the table above)"}, with objective {F_EX:.4f}, return {r_e * 100:.2f}%, volatility {s_e * 100:.2f}%, Sharpe {sh_e:.3f}. The MIQP selects {"the same stocks" if miqp_set == set(sel_gs) else "different stocks"} with free weights and has objective {obj_w(w_miqp):.4f}, return {r_mq_ * 100:.2f}%, volatility {s_mq_ * 100:.2f}%, Sharpe {sh_mq_:.3f}.

**1. Ising and QUBO are the same problem.** The Ising Hamiltonian built with dimod and with the hand-coded formulas agree (differences below 1e-12), the Ising energy equals the QUBO energy on all {2 ** N:,} states (max difference {max_diff:.1e}) and equals the Markowitz objective on the {len(combos):,} feasible states. The exact ground state (sum of spins = {M_FIXED}) is **{fmt(sel_gs)}** at {100 / K:.2f}% each, with energy / objective {F_EX:.4f}; it is the same state as the global minimum over all 2^{N} spin states and the same portfolio as the QUBO exhaustive optimum.

**2. The spin-glass picture.** {(Jp > 0).sum()} of the {len(iu[0])} couplings J_ij are positive (antiferromagnetic) and {(h > 0).sum()} of the {N} local fields h_i are positive; {frustrated} of the {n_tri} triangles are frustrated, on a fully connected graph, with the constraint "exactly k stocks" acting as the fixed magnetisation sum(s) = {M_FIXED}. The penalty alone contributes P/2 = {P / 2:.5f} to every coupling, i.e. {P / 2 / Jp.mean() * 100:.0f}% of the mean coupling ({Jp.mean():.5f}); the covariance part ranges over [{J_cov[iu].min():.5f}, {J_cov[iu].max():.5f}] and is the only source of differences between the couplings (std of J_ij = {Jp.std():.5f}).

**3. Solver behaviour.** With the same budget (1000 reads x 1000 sweeps, {len(SEEDS)} seeds) the D-Wave Ising SA hit the exact optimum in {mi.mean():.1f}% of reads on average (std {mi.std(ddof=1):.1f}, range {mi.min():.1f}-{mi.max():.1f}) against {mq.mean():.1f}% (std {mq.std(ddof=1):.1f}) for the QUBO model with the same seeds; the single run with seed {SEED} gave {hit_i.mean() * 100:.1f}% (Ising) and {hit_q.mean() * 100:.1f}% (QUBO). The best of the 1000 reads was the exact optimum in {int(res_ising["Best read = optimum"].sum())} of {len(SEEDS)} seeds (QUBO: {int(res_qubo["Best read = optimum"].sum())} of {len(SEEDS)}). Feasible reads (exactly {K} stocks): {res_ising["Feasible reads (%)"].mean():.1f}% (Ising) and {res_qubo["Feasible reads (%)"].mean():.1f}% (QUBO). Runtime per run: {t_i:.2f} s (Ising) and {t_q:.2f} s (QUBO).
The own annealers give: Ising with single flips {own_s128:.1f}% (128 x 150) and {own_s1000:.1f}% (1000 x 1000) of reads at the optimum; Ising with single flips plus two-spin moves {own_pair:.1f}% (128 x 150); QUBO with single plus two-bit moves {own_qubo:.1f}% (128 x 150). Changing the **move set** of the own annealer (same budget, 128 x 150) changes the hit rate of the Ising model from {own_s128:.1f}% (single flips) to {own_pair:.1f}% (single flips plus two-spin moves), while changing the **formulation** at the same move set gives {mi.mean():.1f}% (Ising) against {mq.mean():.1f}% (QUBO) for single flips (D-Wave sampler, 1000 x 1000) and {own_pair:.1f}% (Ising) against {own_qubo:.1f}% (QUBO) for single plus swap moves (own annealers, 128 x 150).

**4. Sensitivity.** Sweeps (D-Wave sampler): with 10 sweeps {sw10["Reads at optimum % (mean)"]:.2f}% of reads hit the optimum and {sw10[SEEDCOL]:.0f} of {len(SEEDS)} seeds found it at least once; with 1000 sweeps {sw1000["Reads at optimum % (mean)"]:.2f}% and {sw1000[SEEDCOL]:.0f} of {len(SEEDS)} seeds. Sweeps (own annealer, single flips): {sweeps_txt}. Penalty (D-Wave sampler, 1000 sweeps): reads at the optimum {p05["Reads at optimum % (mean)"]:.2f}% / {p1["Reads at optimum % (mean)"]:.2f}% / {p2["Reads at optimum % (mean)"]:.2f}% and feasible reads {p05["Feasible reads % (mean)"]:.1f}% / {p1["Feasible reads % (mean)"]:.1f}% / {p2["Feasible reads % (mean)"]:.1f}% for 0.5x / 1x / 2x P. Penalty (own annealer, single flips): reads at the optimum {pen["% reads at optimum (mean)"].iloc[0]:.2f}% / {pen["% reads at optimum (mean)"].iloc[1]:.2f}% / {pen["% reads at optimum (mean)"].iloc[2]:.2f}%; the exact ground state of each Hamiltonian {"is the same " + str(K) + " stocks for all three values of P" if pen_same else "changes for at least one value of P (see the table in Section 8.8b)"}.

**5. Weight model.** MIQP (free weights) has objective {obj_w(w_miqp):.4f} against {F_EX:.4f} for the equal-weight Ising / QUBO portfolio (Sharpe {sh_mq_:.3f} against {sh_e:.3f}); this difference comes from the weight model, not from the Ising formulation.

**6. Scale.** At N = {N} the exhaustive enumeration of the {len(combos):,} feasible states takes {t_exact * 1000:.0f} ms in total and the MIQP takes {t_miqp:.2f} s, so this problem is too small to separate the solvers by runtime. The 500-stock notebook repeats the study on a universe where exhaustive enumeration is impossible.
"""))